# Индивидуальная работа №2
## Продвинутый статистический анализ данных и методы машинного обучения

**Тема:** Rail Equipment Accident/Incident Data (Form 54) — Unique Train Accidents (Not at Grade Crossings)

**Студенты:** Doicov Pavel, Iachimenko Alexandr, Morozan Nichita
**Группа:** IA2303  
**Источник данных:** U.S. Department of Transportation — Federal Railroad Administration (FRA)  
`https://data.transportation.gov/Railroads/`


---
# 1. Введение и обоснование темы

## 1.1. Постановка задачи

В данной работе исследуется датасет **Rail Equipment Accident/Incident Data (Form 54)** — официальная база данных США об авариях на железнодорожном транспорте.

Каждый инцидент содержит информацию о:
- типе аварии (сход с рельсов, столкновение и др.)
- скорости и характеристиках поезда
- состоянии путей и погоде
- материальном ущербе и пострадавших

## 1.2. Цели исследования

1. Провести очистку и предобработку данных
2. Выполнить разведочный анализ (EDA) с 20+ визуализациями
3. Построить модели **линейной, логистической и мультиномиальной регрессии**
4. Применить **PCA** для снижения размерности
5. Провести **анализ временных рядов (ARIMA)**
6. Интерпретировать результаты для повышения безопасности перевозок

## 1.3. Описание датасета

| Параметр | Значение |
|----------|----------|
| Источник | U.S. Federal Railroad Administration (FRA) |
| Объём | ~181 737 наблюдений |
| Признаки | 155 колонок |
| Числовые | ~75 признаков |
| Категориальные | ~80 признаков |
| Обновление | Март 2026 |


---
# 2. Предобработка и очистка данных

## 2.1. Загрузка библиотек и данных


In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except ImportError:
    pass

import warnings
warnings.filterwarnings('ignore')

from IPython.display import display

try:
    import plotly.graph_objects as go
    import plotly.io as pio
    PLOTLY_AVAILABLE = True
    try:
        from IPython import get_ipython
        shell = get_ipython().__class__.__name__
        if shell == 'ZMQInteractiveShell':
            pio.renderers.default = 'notebook'
        else:
            pio.renderers.default = 'browser'
    except Exception:
        pio.renderers.default = 'notebook'
except ImportError:
    go = None
    pio = None
    PLOTLY_AVAILABLE = False

from scipy import stats
from scipy.stats import zscore, shapiro, pearsonr
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeCV, ElasticNetCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score, balanced_accuracy_score
)
from sklearn.decomposition import PCA
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.api as sm
from itertools import combinations

plt.rcParams.update({
    'figure.dpi': 110,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
PALETTE = ['#2563EB', '#DC2626', '#16A34A', '#D97706', '#7C3AED', '#0891B2']
sns.set_palette(PALETTE)
np.random.seed(42)

MODEL_NUMERIC_FEATURES = [
    'Train Speed', 'Maximum Speed', 'Gross Tonnage', 'Temperature',
    'Head End Locomotives', 'Loaded Freight Cars', 'Empty Freight Cars',
    'Derailed Loaded Freight Cars', 'Derailed Empty Freight Cars',
    'Derailed Head End Locomotives', 'Derailed Mid Train Manual Locomotives',
    'Derailed Mid Train Remote Locomotives', 'Derailed Rear End Manual Locomotives',
    'Derailed Rear End Remote Locomotives', 'Derailed Cabooses',
    'Track Class', 'Track Density'
]
MODEL_CATEGORICAL_FEATURES = [
    'Accident Type', 'Weather Condition', 'Visibility', 'Track Type',
    'Train Direction', 'Equipment Type', 'Signalization',
    'Method Of Operation', 'Class'
]


def enrich_operational_features(frame):
    data = frame.copy()
    numeric_cols = [c for c in MODEL_NUMERIC_FEATURES + ['Total Damage Cost'] if c in data.columns]
    for col in numeric_cols:
        data[col] = pd.to_numeric(data[col], errors='coerce')

    if {'Loaded Freight Cars', 'Empty Freight Cars'}.issubset(data.columns):
        data['Total Freight Cars'] = data[['Loaded Freight Cars', 'Empty Freight Cars']].sum(axis=1, min_count=1)

    derailed_cols = [
        c for c in [
            'Derailed Loaded Freight Cars', 'Derailed Empty Freight Cars',
            'Derailed Head End Locomotives', 'Derailed Mid Train Manual Locomotives',
            'Derailed Mid Train Remote Locomotives', 'Derailed Rear End Manual Locomotives',
            'Derailed Rear End Remote Locomotives', 'Derailed Cabooses'
        ] if c in data.columns
    ]
    if derailed_cols:
        data['Total Derailed Units'] = data[derailed_cols].sum(axis=1, min_count=1)

    if {'Train Speed', 'Maximum Speed'}.issubset(data.columns):
        data['Speed Utilization'] = data['Train Speed'] / data['Maximum Speed'].replace({0: np.nan})

    if {'Total Derailed Units', 'Total Freight Cars'}.issubset(data.columns):
        data['Derailed Share'] = data['Total Derailed Units'] / data['Total Freight Cars'].replace({0: np.nan})

    if 'Gross Tonnage' in data.columns:
        data['log_Gross Tonnage'] = np.log1p(data['Gross Tonnage'])

    return data


def make_preprocessor(num_cols, cat_cols, scale_numeric=True):
    num_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        num_steps.append(('scaler', StandardScaler()))

    return ColumnTransformer([
        ('num', Pipeline(num_steps), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=0.01, sparse_output=False))
        ]), cat_cols),
    ])


def pretty_feature_name(name):
    clean = name.replace('num__', '').replace('cat__', '')
    clean = clean.replace('_', ' = ')
    return clean



print('Библиотеки и helper-функции загружены успешно')
print(f'Интерактивный Plotly доступен: {PLOTLY_AVAILABLE}')


In [ ]:
import os
from pathlib import Path

# Автоматически находим папку с data.csv и переходим в неё
_data_file = 'data.csv'
if not os.path.exists(_data_file):
    _candidates = [
        Path(r'C:\Users\garik\PycharmProjects\mtsad'),
        Path(r'C:\Users\garik\met statistice'),
        Path.home() / 'PycharmProjects' / 'mtsad',
        Path.home() / 'met statistice',
    ]
    for _d in _candidates:
        if (_d / _data_file).exists():
            os.chdir(_d)
            print(f'Рабочая директория изменена на: {_d}')
            break
    else:
        raise FileNotFoundError(
            'Файл data.csv не найден. '
            'Положите data.csv в ту же папку, что и notebook.'
        )

df = pd.read_csv(_data_file, low_memory=False)

print(f'Размер датасета: {df.shape[0]:,} строк x {df.shape[1]} колонок')
print(f'\nПервые 3 строки:')
df.head(3)


## 2.2. Обработка пропущенных значений

**Стратегия:**
- Признаки с долей пропусков **> 80%** → удаляются (нерелевантны)
- Числовые признаки → заполняются **медианой** (устойчива к выбросам)
- Категориальные признаки → заполняются строкой **"Unknown"**


In [ ]:
#  Анализ пропусков
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2)
}).sort_values('missing_pct', ascending=False)

print("Топ-15 признаков по доле пропусков:")
print(missing.head(15).to_string())

#  Удаление признаков с >80% пропусков
high_missing = missing[missing['missing_pct'] > 80].index.tolist()
df.drop(columns=high_missing, inplace=True)
print(f"\n  Удалено {len(high_missing)} признаков с >80% пропусков")

#  Заполнение пропусков
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include='object').columns

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in cat_cols:
    df[col].fillna('Unknown', inplace=True)

print(f" Оставшиеся пропуски: {df.isna().sum().sum()}")
print(f" Итоговый размер: {df.shape}")


## 2.3. Выявление и обработка выбросов

Используем два метода:
- **IQR** (межквартильный размах): выброс если значение < Q1 − 1.5·IQR или > Q3 + 1.5·IQR
- **Z-score**: выброс если |z| > 3

**Решение:** выбросы **не удаляются**, т.к. крупные аварии с большим ущербом — реальные события, важные для анализа риска. Для моделей применяется логарифмирование.


In [ ]:
key_numeric = [c for c in ['Train Speed','Maximum Speed','Gross Tonnage',
                           'Equipment Damage Cost','Track Damage Cost',
                           'Total Damage Cost','Total Persons Injured',
                           'Total Persons Killed','Temperature']
               if c in df.columns]

rows = []
for col in key_numeric:
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    n_iqr = ((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum()
    n_z   = (np.abs(zscore(s)) > 3).sum()
    rows.append({'Признак': col,
                 'IQR выбросы (%)': round(100*n_iqr/len(s), 2),
                 'Z-score выбросы (%)': round(100*n_z/len(s), 2)})

outlier_df = pd.DataFrame(rows)
print(outlier_df.to_string(index=False))


## 2.4. Кодирование категориальных переменных

Применяем **One-Hot Encoding** для ключевых категориальных признаков.


In [ ]:
cat_to_encode = [c for c in ['Accident Type','Weather Condition','Visibility',
                              'Track Type','Train Direction','Equipment Type']
                 if c in df.columns]

for col in cat_to_encode:
    print(f"{col}: {df[col].nunique()} уникальных значений — {df[col].value_counts().index[:5].tolist()}")


---
# 3. Разведочный анализ данных и визуализации

## 3.1. Описательная статистика числовых признаков


In [ ]:
desc = df[key_numeric].describe().round(2)
print(desc.to_string())


## График 1. Распределение типов аварий (горизонтальная столбчатая диаграмма)

**Что смотрим:** какие типы аварий встречаются чаще всего в датасете.

**Гипотеза:** сходы с рельсов (Derailment) должны составлять подавляющее большинство инцидентов, так как это наиболее частый вид нарушений на железной дороге.

In [ ]:
if 'Accident Type' in df.columns:
    top_types = df['Accident Type'].value_counts().head(8)
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(top_types.index[::-1], top_types.values[::-1], color=PALETTE[:len(top_types)])
    for bar, val in zip(bars, top_types.values[::-1]):
        ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
                f'{val:,}', va='center', fontsize=9)
    ax.set_xlabel('Количество аварий')
    ax.set_title('Рис. 1. Топ-8 типов железнодорожных аварий', fontweight='bold', pad=12)
    plt.tight_layout()
    plt.show()
    top1 = top_types.index[0]
    pct = top_types.values[0] / top_types.values.sum() * 100
    print()
    print(f'Вывод: "{top1}" — самый частый тип аварий.')
    print(f'На его долю приходится {pct:.1f}% от топ-8 типов.')
    print()
    print('Что это значит на практике:')
    print('  Большинство железнодорожных инцидентов — это сходы с рельсов (Derailment).')
    print('  Это происходит, когда колёса вагона или локомотива выходят за пределы рельсов.')
    print('  Причины могут быть разными: дефект пути, слишком высокая скорость на повороте,')
    print('  смещение груза, неисправность оборудования. Именно поэтому регуляторы в первую')
    print('  очередь сосредотачиваются на профилактике сходов, а не других типов аварий.')


## График 2. Распределение общего ущерба (гистограмма)

**Что смотрим:** как распределены суммы ущерба по всем авариям — есть ли "средняя авария" или встречаются редкие дорогостоящие катастрофы.

**Гипотеза:** распределение сильно скошено вправо — большинство аварий дешевые, но редкие катастрофы стоят огромных денег. Это делает среднее значение нерепрезентативным.

In [ ]:
if 'Total Damage Cost' in df.columns:
    data = df['Total Damage Cost'].clip(upper=df['Total Damage Cost'].quantile(0.99))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].hist(data, bins=60, color=PALETTE[0], edgecolor='white', alpha=0.85)
    axes[0].set_title('Линейная шкала')
    axes[0].set_xlabel('Ущерб ($)')
    axes[0].set_ylabel('Частота')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    log_data = np.log1p(df['Total Damage Cost'])
    axes[1].hist(log_data, bins=60, color=PALETTE[1], edgecolor='white', alpha=0.85)
    axes[1].set_title('Логарифмическая шкала (log1p)')
    axes[1].set_xlabel('log(1 + Ущерб)')
    axes[1].set_ylabel('Частота')

    fig.suptitle('Рис. 2. Распределение общего ущерба от аварий', fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    median_val = df['Total Damage Cost'].median()
    mean_val = df['Total Damage Cost'].mean()
    pct_under_10k = (df['Total Damage Cost'] < 10000).mean() * 100
    print()
    print(f'Медиана ущерба: ${median_val:,.0f} | Среднее: ${mean_val:,.0f}')
    print(f'{pct_under_10k:.1f}% аварий нанесли ущерб менее $10,000')
    print()
    print('Что это значит на практике:')
    print('  Левый график (линейная шкала) показывает, что почти все аварии "сжаты" у нуля —')
    print('  большинство инцидентов относительно дешевые, но шкала вынуждена "тянуться" к')
    print('  редким катастрофам стоимостью миллионы долларов.')
    print()
    print('  Правый график (логарифмическая шкала) "растягивает" мелкие значения и "сжимает"')
    print('  крупные — теперь распределение выглядит почти нормальным (колоколообразным).')
    print('  Именно поэтому во всех регрессионных моделях мы используем log(1+ущерб) как цель:')
    print('  это делает модель значительно более точной и устойчивой.')
    print()
    print('  Среднее значение в ~3 раза выше медианы — это классический признак')
    print('  "тяжёлого хвоста": несколько дорогостоящих аварий сильно завышают среднее.')


## График 3. Динамика числа аварий по годам (временной ряд)

**Что смотрим:** меняется ли количество железнодорожных аварий со временем и в каком направлении.

**Гипотеза:** число аварий снижается благодаря улучшению технологий безопасности, ужесточению регулирования и модернизации инфраструктуры.

In [ ]:
year_col = 'Year' if 'Year' in df.columns else ('Accident Year' if 'Accident Year' in df.columns else None)
if year_col:
    year_values = pd.to_numeric(df[year_col], errors='coerce')
    if year_values.dropna().max() <= 99:
        year_values = pd.Series(
            np.where(year_values.isna(), np.nan,
                     np.where(year_values <= 30, year_values + 2000, year_values + 1900)),
            index=df.index,
        )
    year_values = year_values[year_values.between(1975, 2025)]
    yearly = year_values.astype(int).value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.fill_between(yearly.index, yearly.values, alpha=0.25, color=PALETTE[0])
    ax.plot(yearly.index, yearly.values, color=PALETTE[0], linewidth=2.2, marker='o', markersize=3)
    ax.set_xlabel('Год')
    ax.set_ylabel('Число аварий')
    ax.set_title('Рис. 3. Количество железнодорожных аварий по годам (1975-2025)',
                 fontweight='bold', pad=12)
    z = np.polyfit(yearly.index, yearly.values, 1)
    p = np.poly1d(z)
    ax.plot(yearly.index, p(yearly.index), '--', color=PALETTE[1], linewidth=1.5, label='Тренд')
    ax.legend()
    plt.tight_layout()
    plt.show()

    peak_year = yearly.idxmax()
    min_year  = yearly.idxmin()
    decline_pct = (yearly[peak_year] - yearly[min_year]) / yearly[peak_year] * 100
    print()
    print(f'Пиковый год: {peak_year} ({yearly[peak_year]:,} аварий)')
    print(f'Минимум в выборке: {min_year} ({yearly[min_year]:,} аварий)')
    print(f'Снижение от пика: {decline_pct:.1f}%')
    print()
    print('Что это значит на практике:')
    print('  Красная пунктирная линия тренда показывает общее направление изменений.')
    print('  Явный нисходящий тренд означает, что железнодорожная отрасль США стала')
    print('  значительно безопаснее с 1970-80-х годов до сегодняшнего дня.')
    print()
    print('  Это объясняется несколькими факторами:')
    print('  1. Технология: улучшенные системы автоматического торможения (PTC),')
    print('     современные датчики состояния пути, GPS-мониторинг.')
    print('  2. Регулирование: после крупных катастроф (например, Чикаго, 1972 г.) приняты')
    print('     строгие стандарты безопасности.')
    print('  3. Инфраструктура: масштабная замена рельсов и шпал.')
    print('  Несмотря на это, статистика не нулевая: аварии по-прежнему происходят,')
    print('  и задача исследования — понять, что ими движет.')


## График 4. Сезонность аварий по месяцам (столбчатая диаграмма)

**Что смотрим:** в какие месяцы года происходит больше всего железнодорожных аварий — есть ли выраженная сезонность.

**Гипотеза:** зимние месяцы должны показывать повышенную аварийность из-за снега, льда и низких температур, влияющих на состояние пути и видимость.

In [ ]:
month_col = 'Accident Month' if 'Accident Month' in df.columns else None
if month_col and month_col in df.columns:
    monthly = df[month_col].value_counts().sort_index()
    months_ru = ['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн',
                 'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек']
    monthly.index = [months_ru[m - 1] if 1 <= m <= 12 else str(m) for m in monthly.index]

    peak = monthly.idxmax()
    low  = monthly.idxmin()

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = [PALETTE[1] if m == peak else PALETTE[0] for m in monthly.index]
    ax.bar(monthly.index, monthly.values, color=colors, edgecolor='white')
    ax.set_xlabel('Месяц')
    ax.set_ylabel('Число аварий')
    ax.set_title('Рис. 4. Сезонность железнодорожных аварий по месяцам', fontweight='bold', pad=12)
    plt.tight_layout()
    plt.show()
    ratio = monthly.max() / monthly.min()
    print()
    print(f'Пиковый месяц: {peak} ({monthly.max():,} аварий)')
    print(f'Самый спокойный: {low} ({monthly.min():,} аварий)')
    print(f'Разрыв пик/минимум: x{ratio:.2f}')
    print()
    print('Что это значит на практике:')
    print('  Если сезонность выражена, это говорит о том, что условия окружающей среды')
    print('  существенно влияют на аварийность. В зимние месяцы снег и лёд:')
    print('  - ухудшают видимость для машинистов;')
    print('  - создают риск замерзания стрелочных переводов;')
    print('  - снижают коэффициент трения между колесом и рельсом.')
    print('  Летом высокие температуры могут вызывать деформацию рельсов (thermal kink).')
    print('  Если же пик приходится на другой период — возможно, важную роль играет')
    print('  объём перевозок: высокий трафик всегда означает больший риск инцидентов.')


## График 5. Аварии по типам пути (Track Type)

**Что смотрим:** на путях какого типа чаще всего происходят аварии и как это соотносится с долей каждого типа в общей сети.

**Гипотеза:** главные линии (Main Track) дают больше аварий в абсолютных числах, но второстепенные и сортировочные пути могут быть опаснее на единицу длины.

In [ ]:
if 'Track Type' in df.columns:
    track_data = df['Track Type'].value_counts()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].barh(track_data.index, track_data.values,
                 color=PALETTE[:len(track_data)], edgecolor='white')
    axes[0].set_xlabel('Число аварий')
    axes[0].set_title('Число аварий по типу пути', fontweight='bold')

    axes[1].pie(track_data.values, labels=track_data.index, autopct='%1.1f%%',
                colors=PALETTE[:len(track_data)], startangle=90)
    axes[1].set_title('Доля каждого типа', fontweight='bold')

    fig.suptitle('Рис. 5. Аварии по типам железнодорожного пути', fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    top_track = track_data.index[0]
    top_pct   = track_data.values[0] / track_data.values.sum() * 100
    print()
    print(f'Лидер: "{top_track}" — {top_pct:.1f}% всех аварий.')
    print()
    print('Что это значит на практике:')
    print('  Тип пути напрямую связан с интенсивностью движения и стандартами обслуживания.')
    print('  Главные линии (Main Track) перевозят основной объём грузов и пассажиров,')
    print('  поэтому они хорошо обслуживаются — но из-за высокой загруженности статистически')
    print('  дают больше аварий в абсолютных числах.')
    print('  Ярдовые и сортировочные пути (Yard Track) работают на малых скоростях, но')
    print('  манёвровые операции — специфический источник рисков.')
    print('  Для сравнения: если разделить число аварий на суммарную длину каждого типа пути,')
    print('  картина может кардинально измениться.')


## График 6. Ущерб по типу аварии (боксплот)

**Что смотрим:** насколько сильно различается стоимость аварии в зависимости от её типа — сход с рельсов, столкновение, пожар и т.д.

**Гипотеза:** столкновения и пожары должны давать более высокий медианный ущерб, чем сходы с рельсов, хотя последние встречаются намного чаще.

In [ ]:
if 'Accident Type' in df.columns and 'Total Damage Cost' in df.columns:
    top4 = df['Accident Type'].value_counts().head(4).index
    sub = df[df['Accident Type'].isin(top4)].copy()
    sub['log_damage'] = np.log1p(sub['Total Damage Cost'])

    fig, ax = plt.subplots(figsize=(10, 5))
    sub.boxplot(column='log_damage', by='Accident Type', ax=ax,
                patch_artist=True,
                boxprops=dict(facecolor=PALETTE[0], alpha=0.6),
                medianprops=dict(color='red', linewidth=2))
    ax.set_xlabel('Тип аварии')
    ax.set_ylabel('log(1 + Ущерб)')
    ax.set_title('Рис. 6. Распределение ущерба по типу аварии', fontweight='bold', pad=12)
    plt.suptitle('')
    plt.tight_layout()
    plt.show()
    medians = sub.groupby('Accident Type')['log_damage'].median().sort_values(ascending=False)
    print()
    print('Медианный log(ущерб) по типам аварий:')
    for atype, med in medians.items():
        print(f'  {atype:<30} {med:.3f}  (~${np.expm1(med):,.0f})')
    print()
    print('Что это значит на практике:')
    print('  Боксплот (ящик с усами) показывает не одно значение, а разброс ущерба:')
    print('  - Линия внутри ящика — медиана (типичная авария этого типа);')
    print('  - Ящик — диапазон, в котором лежат 50% аварий;')
    print('  - Усы — разумный диапазон значений;')
    print('  - Точки вне усов — исключительно дорогостоящие единичные события.')
    print()
    print('  Широкий ящик и многочисленные "точки-выбросы" говорят о том, что ущерб')
    print('  очень сильно варьируется даже внутри одного типа аварии.')
    print('  Это важный сигнал: тип аварии сам по себе не полностью объясняет стоимость,')
    print('  важен ещё и контекст (скорость, тоннаж, тип пути, погода).')


## График 7. Корреляционная матрица числовых признаков (тепловая карта)

**Что смотрим:** какие пары числовых переменных сильно взаимосвязаны между собой. Это важно перед построением регрессии: если два признака почти одинаковы, одного из них достаточно.

**Что такое корреляция:** коэффициент от -1 до +1. Значение +1 означает: растет один — растет другой. -1 означает: растет один — падает другой. 0 — переменные не связаны.

In [ ]:
corr_cols = [c for c in ['Train Speed', 'Maximum Speed', 'Gross Tonnage',
                           'Equipment Damage Cost', 'Track Damage Cost',
                           'Total Damage Cost', 'Total Persons Killed',
                           'Temperature', 'Head End Locomotives',
                           'Loaded Freight Cars', 'Total Freight Cars',
                           'Derailed Loaded Freight Cars', 'Total Derailed Units',
                           'Track Density']
             if c in df.columns]

# Принудительно переводим все столбцы в числовые (строки вроде '.' заменяются на NaN)
sub_corr = df[corr_cols].apply(pd.to_numeric, errors='coerce').dropna()
corr_matrix = sub_corr.corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', annot_kws={'size': 8},
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Рис. 7. Корреляционная матрица числовых признаков', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

# Топ-5 пар по абсолютной корреляции (вне главной диагонали)
corr_pairs = []
for i, c1 in enumerate(corr_cols):
    for c2 in corr_cols[i+1:]:
        if c1 in corr_matrix.columns and c2 in corr_matrix.columns:
            r = corr_matrix.loc[c1, c2]
            corr_pairs.append((abs(r), r, c1, c2))
corr_pairs.sort(reverse=True)

print()
print('Топ-5 наиболее скоррелированных пар признаков:')
for _, r, c1, c2 in corr_pairs[:5]:
    print(f'  {c1}  <->  {c2}: r = {r:.3f}')

print()
print('Что это значит на практике:')
print('  Синие клетки (близко к -1): признаки движутся в противоположных направлениях.')
print('  Красные клетки (близко к +1): признаки растут и падают вместе.')
print()
print('  Самая сильная корреляция — между компонентами ущерба:')
print('  Equipment Damage Cost и Track Damage Cost оба входят в Total Damage Cost.')
print('  Поэтому в регрессионных моделях мы их ИСКЛЮЧАЕМ: использование частей')
print('  для предсказания итоговой суммы — это не честный прогноз, а тавтология.')
print()
print('  Высокая корреляция между Train Speed и Maximum Speed — тоже ожидаема:')
print('  поезда обычно идут близко к максимально допустимой скорости.')
print('  Это называется мультиколлинеарность — её учитываем при построении моделей.')


## График 8. Диаграмма рассеяния: скорость поезда vs ущерб

**Что смотрим:** есть ли прямая связь между скоростью поезда в момент аварии и суммой ущерба — иными словами, быстрые поезда попадают в более дорогостоящие аварии?

**Гипотеза:** более высокая скорость означает большую кинетическую энергию, что должно приводить к большему разрушению и, соответственно, большему ущербу.

In [ ]:
if 'Train Speed' in df.columns and 'Total Damage Cost' in df.columns:
    sub = df[(df['Train Speed'] > 0) & (df['Total Damage Cost'] > 0)].sample(
        min(5000, len(df)), random_state=42)
    log_speed = np.log1p(sub['Train Speed'])
    log_damage = np.log1p(sub['Total Damage Cost'])
    r_val, p_val = pearsonr(log_speed, log_damage)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(log_speed, log_damage, alpha=0.25, s=12, color=PALETTE[0])
    m, b = np.polyfit(log_speed, log_damage, 1)
    x_line = np.linspace(log_speed.min(), log_speed.max(), 100)
    ax.plot(x_line, m * x_line + b, color=PALETTE[1], linewidth=2.2,
            label=f'Линия регрессии (r = {r_val:.3f})')
    ax.set_xlabel('log(1 + Скорость поезда, миль/ч)')
    ax.set_ylabel('log(1 + Ущерб, $)')
    ax.set_title('Рис. 8. Скорость поезда vs Ущерб от аварии', fontweight='bold', pad=12)
    ax.legend()
    plt.tight_layout()
    plt.show()
    sig = 'статистически значима' if p_val < 0.05 else 'статистически незначима'
    print()
    print(f'Коэффициент корреляции Пирсона r = {r_val:.3f} (p = {p_val:.2e})')
    print(f'Связь {sig}')
    print()
    print('Что это значит на практике:')
    print('  Каждая точка на графике — одна авария. Мы используем логарифмическую шкалу,')
    print('  чтобы "уплотнить" экстремальные значения и увидеть общую картину.')
    print()
    print('  Наклонная линия тренда показывает: в среднем, при увеличении скорости')
    print('  ущерб действительно растет. Это физически логично — более быстрое столкновение')
    print('  высвобождает больше кинетической энергии.')
    print()
    print('  Однако коэффициент корреляции умеренный (не близкий к 1). Это значит:')
    print('  - Скорость важна, но не является единственным фактором;')
    print('  - Много дорогостоящих аварий происходит и на низкой скорости;')
    print('  - Сорт груза, тип пути, количество сошедших вагонов добавляют не меньше.')


## График 9. Аварии по штатам США (топ-15)

**Что смотрим:** в каких штатах происходит наибольшее число железнодорожных аварий и насколько велика разница между лидерами и остальными.

**Гипотеза:** штаты с развитой железнодорожной сетью (Техас, Иллинойс, Калифорния) и крупными грузовыми коридорами должны лидировать по числу инцидентов.

In [ ]:
if 'State Name' in df.columns:
    top_states = df['State Name'].value_counts().head(15)

    fig, ax = plt.subplots(figsize=(10, 6))
    colors = [PALETTE[1] if i == 0 else PALETTE[0] for i in range(len(top_states))]
    ax.barh(top_states.index[::-1], top_states.values[::-1], color=colors[::-1], edgecolor='white')
    for i, (val, name) in enumerate(zip(top_states.values[::-1], top_states.index[::-1])):
        ax.text(val + 100, i, f'{val:,}', va='center', fontsize=8)
    ax.set_xlabel('Количество аварий')
    ax.set_title('Рис. 9. Топ-15 штатов США по числу железнодорожных аварий', fontweight='bold', pad=12)
    plt.tight_layout()
    plt.show()
    print()
    print(f'Лидер: {top_states.index[0]} ({top_states.values[0]:,} аварий)')
    print(f'15-е место: {top_states.index[-1]} ({top_states.values[-1]:,} аварий)')
    print(f'Разница: x{top_states.values[0]/top_states.values[-1]:.1f}')
    print()
    print('Что это значит на практике:')
    print('  Число аварий в штате определяется несколькими факторами:')
    print('  1. Протяженность железнодорожной сети: больше путей — больше рисков.')
    print('  2. Объём грузовых перевозок: промышленные штаты Среднего Запада')
    print('     (Иллинойс, Огайо, Техас) активно используют ж/д для грузов.')
    print('  3. Возраст инфраструктуры: старые пути в восточных штатах требуют')
    print('     более частого обслуживания.')
    print()
    print('  Важно: высокое число аварий не означает "опасный штат" для пассажиров.')
    print('  Большинство инцидентов — это грузовые сходы с рельсов без пострадавших.')
    print('  Для оценки реального риска нужно нормировать на объём перевозок.')


## График 10. Аварии по погодным условиям (столбчатая диаграмма)

**Что смотрим:** при каких погодных условиях чаще всего фиксируются железнодорожные аварии — в плохую или в хорошую погоду?

**Важная оговорка:** большое число аварий при ясной погоде не означает, что ясная погода опасна. Она просто самая частая. Чтобы сделать честный вывод, нужно нормировать на долю каждого типа погоды в общем времени.

In [ ]:
if 'Weather Condition' in df.columns:
    weather = df['Weather Condition'].value_counts()

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(weather.index, weather.values, color=PALETTE[:len(weather)], edgecolor='white')
    ax.set_xlabel('Погодные условия')
    ax.set_ylabel('Число аварий')
    ax.set_title('Рис. 10. Распределение аварий по погодным условиям', fontweight='bold', pad=12)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
    clear_pct = weather.get('Clear', weather.iloc[0]) / weather.sum() * 100
    top_weather = weather.index[0]
    print()
    print(f'Самые частые условия: "{top_weather}" ({clear_pct:.1f}% всех аварий)')
    print()
    print('Что это значит на практике:')
    print('  Кажется парадоксальным: большинство аварий происходит при ясной погоде.')
    print('  Но это не значит, что ясная погода опасна!')
    print()
    print('  Причина проста: ясная погода — самая распространенная. Если, например,')
    print('  в году 70% дней солнечных, то и 70% аварий будут "при ясной погоде"')
    print('  — даже если в плохую погоду авария случается чаще за один день.')
    print()
    print('  Чтобы честно оценить влияние погоды, нужна нормировка:')
    print('  (число аварий при данной погоде) / (число дней с такой погодой).')
    print('  Тогда, скорее всего, снег и туман окажутся значительно опаснее ясного неба.')


---
# Регрессионный анализ

---

## Что такое регрессия и зачем она нужна?

**Регрессия** — это метод, который находит математическую связь между
**входными признаками** (X) и **целевой переменной** (y).

| Тип | Цель | Пример в нашем проекте |
|-----|------|------------------------|
| **Линейная регрессия** | Предсказать число | Сколько $ ущерба от аварии? |
| **Логистическая регрессия** | Класс 0 или 1 | Будут ли жертвы? |
| **Мультиномиальная** | Один из K классов | Тип пути: Main/Yard/Industry? |

---

## Линейная регрессия — основы

### Формула модели:
```
ŷ = β₀ + β₁·x₁ + β₂·x₂ + ... + βₖ·xₖ
```
- **ŷ** — прогноз (у нас: логарифм ущерба)
- **β₀** — свободный член (значение y когда все x = 0)
- **β₁...βₖ** — коэффициенты: насколько меняется y при +1 единице xᵢ
- **x₁...xₖ** — признаки (скорость, тоннаж, температура...)

### Как обучается — Метод Наименьших Квадратов (OLS):
Модель ищет коэффициенты β, минимизируя сумму квадратов ошибок:
```
min Σ(yᵢ - ŷᵢ)²
```
Аналитическое решение: **β̂ = (XᵀX)⁻¹Xᵀy**

### Метрики качества — как читать результаты:

| Метрика | Формула | Что означает | Лучше когда |
|---------|---------|--------------|-------------|
| **R²** | 1 − SS_res/SS_tot | Доля объяснённой дисперсии | Ближе к 1 |
| **RMSE** | √(Σ(y−ŷ)²/n) | Средняя квадратичная ошибка | Меньше |
| **MAE** | Σ|y−ŷ|/n | Средняя абсолютная ошибка | Меньше |

**R² = 0.3** → модель объясняет 30% разброса (остальное — шум/неизвестные факторы)
**R² = 0.8** → хорошо: 80% объяснено

> **Важно:** После `StandardScaler` все признаки имеют μ=0, σ=1.
> Тогда β = 0.5 означает: «рост признака на 1σ → рост y на 0.5σ»

---

## OLS через statsmodels — что читать в сводке

| Показатель | Что читать |
|------------|-----------|
| **coef** | Значение коэффициента β |
| **std err** | Стандартная ошибка — точность оценки β |
| **t / P>|t|** | t-статистика и p-значение: p < 0.05 → признак значим |
| **[0.025, 0.975]** | 95% доверительный интервал для β |
| **R-squared** | Доля объяснённой дисперсии |
| **F-statistic** | Тест значимости всей модели (p < 0.05 → модель работает) |
| **AIC / BIC** | Качество модели с штрафом за сложность (меньше = лучше) |

### Q-Q Plot остатков:
Линейная регрессия предполагает, что остатки ε ~ N(0, σ²).
На Q-Q Plot: точки вдоль диагонали → нормальность подтверждена.

---

## Регуляризация: Ridge и Lasso

**Проблема OLS:** при многих признаках возможно переобучение.
**Решение:** добавить штраф за большие коэффициенты.

### Ridge (L2):
```
min { Σ(y−ŷ)² + λ·Σβⱼ² }
```
→ Уменьшает ВСЕ коэффициенты, но не обнуляет. Хорош при мультиколлинеарности.

### Lasso (L1):
```
min { Σ(y−ŷ)² + λ·Σ|βⱼ| }
```
→ **Обнуляет** малозначимые β — встроенный отбор признаков!

**λ (alpha в sklearn):** чем больше → тем сильнее штраф → проще модель.

---

## Логистическая регрессия — классификация через вероятности

Несмотря на слово "регрессия" — это **алгоритм классификации**.

### Сигмоидная функция:
```
P(Y=1|x) = σ(z) = 1 / (1 + e^(-z))    где z = β₀ + β₁x₁ + ...
```
Выход — **вероятность** принадлежности к классу 1.
Если P > 0.5 → предсказываем класс 1, иначе класс 0.

### Матрица ошибок (Confusion Matrix):
```
                 Предсказано: 0    Предсказано: 1
Реальный: 0   [  TN (верно!)       FP (ложная тревога) ]
Реальный: 1   [  FN (пропуск!)     TP (верно!)         ]
```
**Диагональ (TN, TP)** — правильные ответы → хотим максимизировать
**FN** — пропущенные жертвы → самая опасная ошибка в задачах безопасности!

### Метрики классификации:

| Метрика | Формула | Смысл |
|---------|---------|-------|
| **Accuracy** | (TP+TN)/(P+N) | Доля правильных ответов |
| **Precision** | TP/(TP+FP) | Из предсказанных "1" — сколько верных |
| **Recall** | TP/(TP+FN) | Из реальных "1" — сколько нашли |
| **F1-score** | 2·P·R/(P+R) | Баланс Precision и Recall |
| **ROC-AUC** | Площадь под ROC | 0.5=случайно, 1.0=идеально |

### Несбалансированные классы — частая проблема:
Если жертвы в 5% случаев → наивная модель: "нет жертв" всегда → Accuracy = 95%
→ **бесполезная модель!**
**Решение:** `class_weight='balanced'` — увеличивает штраф за ошибки на редком классе.

---

## Мультиномиальная логистическая регрессия (K классов)

Расширение на **K > 2** классов через функцию **Softmax**:
```
P(Y=k|x) = exp(xᵀβₖ) / Σⱼ exp(xᵀβⱼ)
```
Для каждого класса — своя линейная комбинация, softmax нормирует в вероятности (сумма = 1).

---

## Как выбрать лучшую модель?

1. **R² / AUC** — основной показатель качества
2. **Cross-validation (CV)** — честная оценка, не переобучение
3. **Кривая обучения** — train >> validation → переобучение
4. **F1 vs Accuracy** — при дисбалансе классов F1 важнее Accuracy
5. **Бизнес-задача** — Recall важнее Precision при высокой цене FN

---

---
# 4. Модели регрессии и классификации

## 4.1. Формулировка задачи

Мы рассматриваем три типа регрессионных задач:

| Задача | Тип | Целевая переменная |
|--------|-----|--------------------|
| 1 | **Линейная регрессия (множественная)** | `log(Total Damage Cost)` — непрерывная |
| 2 | **Логистическая регрессия (бинарная)** | Ущерб выше/ниже медианы (`high_damage`) |
| 3 | **Мультиномиальная логистическая регрессия** | Тип пути: Main / Yard / Industry |

---

## 4.2. Линейная (множественная) регрессия

### Теоретическая база

Модель множественной линейной регрессии:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \ldots + \beta_k x_k + \varepsilon$$

Где:
- $\hat{y}$ — прогнозируемое значение (логарифм ущерба)
- $\beta_0$ — свободный член (intercept)
- $\beta_i$ — коэффициенты при каждом признаке
- $x_i$ — значения признаков (скорость, тоннаж и т.д.)
- $\varepsilon \sim \mathcal{N}(0, \sigma^2)$ — случайная ошибка

**Метод оценки коэффициентов** — Метод наименьших квадратов (OLS):

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

**Метрики качества:**
- $R^2 = 1 - \dfrac{SS_{res}}{SS_{tot}}$ — доля объяснённой дисперсии
- $RMSE = \sqrt{\dfrac{1}{n}\sum(y_i - \hat{y}_i)^2}$ — средняя квадратичная ошибка
- $MAE = \dfrac{1}{n}\sum|y_i - \hat{y}_i|$ — средняя абсолютная ошибка


In [ ]:
model_df = enrich_operational_features(df)

reg_df = model_df[model_df['Total Damage Cost'].fillna(0) > 0].copy()
reg_df['log_total_damage'] = np.log1p(reg_df['Total Damage Cost'])

reg_num_features = [
    c for c in [
        'Train Speed', 'Maximum Speed', 'Gross Tonnage', 'log_Gross Tonnage', 'Temperature',
        'Head End Locomotives', 'Loaded Freight Cars', 'Empty Freight Cars', 'Total Freight Cars',
        'Derailed Loaded Freight Cars', 'Derailed Empty Freight Cars', 'Total Derailed Units',
        'Track Class', 'Track Density', 'Speed Utilization', 'Derailed Share'
    ] if c in reg_df.columns
]
reg_cat_features = [
    c for c in [
        'Accident Type', 'Weather Condition', 'Visibility', 'Track Type',
        'Train Direction', 'Equipment Type', 'Signalization', 'Method Of Operation'
    ] if c in reg_df.columns
]

X_reg = reg_df[reg_num_features + reg_cat_features].copy()
y_reg = reg_df['log_total_damage'].copy()

reg_preprocessor = make_preprocessor(reg_num_features, reg_cat_features, scale_numeric=True)

X_reg_train_raw, X_reg_test_raw, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

X_train = reg_preprocessor.fit_transform(X_reg_train_raw)
X_test = reg_preprocessor.transform(X_reg_test_raw)
y_train = y_reg_train
y_test = y_reg_test
reg_feature_names = reg_preprocessor.get_feature_names_out()

leakage_features = [c for c in ['Equipment Damage Cost', 'Track Damage Cost'] if c in df.columns]

print('Подготовка данных для модели ущерба')
print(f'  Строк с положительным ущербом: {len(reg_df):,}')
print(f'  Обучающая выборка: {X_train.shape[0]:,}')
print(f'  Тестовая выборка:  {X_test.shape[0]:,}')
print()
print('Что именно проверяем:')
print('  Можно ли по характеристикам поезда, пути и типа аварии заранее объяснить размер ущерба.')
print('  Целевая переменная: log(1 + Total Damage Cost).')
print()
print('Почему этот набор признаков лучше предыдущего:')
print('  Мы специально НЕ используем Equipment Damage Cost и Track Damage Cost.')
print('  Эти величины входят в Total Damage Cost и давали бы модели почти готовый ответ.')
if leakage_features:
    print(f'  Исключены признаки с утечкой цели: {leakage_features}')
print()
print(f'Числовые признаки: {reg_num_features}')
print(f'Категориальные признаки: {reg_cat_features}')


In [ ]:
reg_models = {
    'OLS (без штрафа)': LinearRegression(),
    'Ridge (стабильная)': RidgeCV(alphas=np.logspace(-3, 3, 10)),
    'ElasticNet (отбор)': ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
        alphas=np.logspace(-3, 1, 8),
        max_iter=5000,
        cv=3,
        random_state=42
    ),
}

trained_reg_models = {}
results_reg = {}

for name, model in reg_models.items():
    model.fit(X_train, y_train)
    trained_reg_models[name] = model
    y_pred = model.predict(X_test)
    cv_pipe = Pipeline([
        ('preprocessor', make_preprocessor(reg_num_features, reg_cat_features, scale_numeric=True)),
        ('model', model)
    ])
    cv_r2 = cross_val_score(cv_pipe, X_reg, y_reg, cv=3, scoring='r2', n_jobs=1)
    results_reg[name] = {
        'R² (test)': round(r2_score(y_test, y_pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
        'MAE': round(mean_absolute_error(y_test, y_pred), 4),
        'CV R² mean': round(cv_r2.mean(), 4),
        'CV R² std': round(cv_r2.std(), 4),
    }

results_df = pd.DataFrame(results_reg).T.sort_values('CV R² mean', ascending=False)
best_candidate = results_df.index[0]
ridge_gap = abs(results_df.loc['Ridge (стабильная)', 'CV R² mean'] - results_df.iloc[0]['CV R² mean'])
reg_model_name = 'Ridge (стабильная)' if ridge_gap <= 0.005 else best_candidate
lr = trained_reg_models[reg_model_name]
y_pred_lr = lr.predict(X_test)

r2 = r2_score(y_test, y_pred_lr)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae = mean_absolute_error(y_test, y_pred_lr)

coef_series = pd.Series(lr.coef_, index=reg_feature_names)
coef_filtered = coef_series[~coef_series.index.str.contains('infrequent_sklearn')]
coef_df = (
    coef_filtered.rename_axis('raw_feature')
    .reset_index(name='Коэффициент')
    .assign(**{
        'Признак': lambda d: d['raw_feature'].map(pretty_feature_name),
        'Абс. значение': lambda d: d['Коэффициент'].abs()
    })
    .sort_values('Абс. значение', ascending=False)
)

ridge_coef_display_df = coef_df[
    ~coef_df['Признак'].str.contains('Class =', regex=False)
].head(12).copy()
coef_compare_features = ridge_coef_display_df['Признак'].tolist()[:8]

print('=' * 70)
print('РЕГРЕССИЯ ДЛЯ ПРОГНОЗА УЩЕРБА')
print('=' * 70)
print(results_df.to_string())
print()
print(f'Выбранная модель: {reg_model_name}')
print(f'  R² на тесте: {r2:.4f}')
print(f'  RMSE:        {rmse:.4f}')
print(f'  MAE:         {mae:.4f}')
print()
print('Как это читать без математики:')
print(f'  Модель объясняет примерно {r2*100:.1f}% различий в размере ущерба.')
print('  Это не идеальный результат, но он реалистичен для аварийных данных:')
print('  на итоговый ущерб влияют и неполно наблюдаемые факторы, и редкие экстремальные события.')
print(f'  Типичная ошибка по RMSE в лог-шкале соответствует примерно множителю x{np.exp(rmse):.2f} в реальных долларах.')
print()
print('Главный вывод по моделям:')
print('  OLS, Ridge и ElasticNet дали почти одинаковое качество.')
print('  Поэтому разумно выбрать Ridge: она остаётся интерпретируемой и устойчивее к шуму и мультиколлинеарности.')
print()
print('Самые заметные факторы у выбранной модели:')
print(ridge_coef_display_df[['Признак', 'Коэффициент']].to_string(index=False))


## График 11. Коэффициенты регрессии Ridge

**Что смотрим:** какие признаки сильнее всего связаны с размером ущерба и в каком направлении.

**Как читать:**
- Синяя полоса вправо (положительный коэффициент): рост этого признака связан с ростом ущерба
- Красная полоса влево (отрицательный коэффициент): рост этого признака связан со снижением ущерба
- Длина полосы показывает силу влияния

**Важно:** коэффициенты стандартизированы — их можно напрямую сравнивать между собой.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = ridge_coef_display_df.iloc[::-1]
colors = [PALETTE[1] if c < 0 else PALETTE[0] for c in plot_df['Коэффициент']]
ax.barh(plot_df['Признак'], plot_df['Коэффициент'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Стандартизованный коэффициент')
ax.set_title('Рис. 11. Влияние каждого фактора на размер ущерба (Ridge)', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

print()
print('Вывод:')
print('  Этот график показывает вклад каждого признака при фиксированных остальных.')
print('  Самые длинные синие полосы (вправо) — факторы, увеличивающие ущерб.')
print('  Самые длинные красные полосы (влево) — факторы, уменьшающие ущерб.')
print()
print('  Что означает каждый из ключевых признаков на практике:')
print('  - "Derailed Loaded Freight Cars" (гружёные сошедшие вагоны): каждый')
print('    дополнительный сошедший гружёный вагон добавляет заметный вклад в ущерб.')
print('    Это логично — гружёный вагон сложнее поднять и он может повредить путь.')
print('  - "Maximum Speed" (допустимая скорость): участки с высоким ограничением скорости')
print('    обычно являются главными коридорами с более тяжёлыми поездами и большим трафиком.')
print('  - "Gross Tonnage" (тоннаж): тяжелее состав — больше разрушений при аварии.')
print()
print('  Для обычного читателя: коэффициент +0.3 означает, что при увеличении')
print('  данного признака на одно стандартное отклонение ущерб вырастает на 0.3')
print('  в логарифмической шкале, что примерно соответствует росту на 35% в долларах.')


## График 12. Реальные vs Предсказанные значения

**Что смотрим:** насколько точно модель предсказывает реальный ущерб.

**Как читать:**
- Слева: каждая точка — одна авария. Красная пунктирная линия — идеальный прогноз (предсказание = реальность). Чем плотнее точки к этой линии, тем точнее модель.
- Справа: "остатки" (разница между реальностью и прогнозом). Хаотичное облако вокруг нуля = хорошо. Если видна структура (изгиб, расширение) = модель систематически ошибается.

In [ ]:
sample_idx = np.random.choice(len(y_test), min(2500, len(y_test)), replace=False)
y_t_sample = np.array(y_test)[sample_idx]
y_p_sample = y_pred_lr[sample_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_t_sample, y_p_sample, alpha=0.3, s=15, color=PALETTE[0])
mn = min(y_t_sample.min(), y_p_sample.min())
mx = max(y_t_sample.max(), y_p_sample.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='идеальное совпадение')
axes[0].set_xlabel('Реальный log(1 + ущерб)')
axes[0].set_ylabel('Прогноз модели')
axes[0].set_title(f'Реальные vs прогнозные значения, R2 = {r2:.3f}')
axes[0].legend()

residuals = y_t_sample - y_p_sample
axes[1].scatter(y_p_sample, residuals, alpha=0.3, s=15, color=PALETTE[2])
axes[1].axhline(0, color='red', linewidth=2, linestyle='--')
axes[1].set_xlabel('Прогноз модели')
axes[1].set_ylabel('Остаток (реальность - прогноз)')
axes[1].set_title('График остатков')

fig.suptitle('Рис. 12. Проверка качества регрессионной модели', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print()
print(f'R2 (коэффициент детерминации) = {r2:.4f}')
print(f'RMSE = {rmse:.4f} | MAE = {mae:.4f}')
print()
print('Что это значит на практике:')
print('  Левый график: идеальная модель имела бы все точки строго на красной линии.')
print('  Наша модель показывает типичное "веретено" — хорошо работает для средних')
print('  значений ущерба, но ошибается на экстремальных (очень дорогих авариях).')
print()
print(f'  R2 = {r2:.3f} означает: модель объясняет {r2*100:.1f}% различий в ущербе.')
print('  Оставшиеся {:.1f}% объясняются факторами, которых нет в датасете:'.format((1-r2)*100))
print('  состояние конкретного участка пути, квалификация машиниста, время реакции')
print('  экстренных служб, качество груза и т.д.')
print()
print('  Правый график (остатки): точки распределены хаотично вокруг нуля —')
print('  это хороший знак. Систематической ошибки в одну сторону нет.')
print('  Широкое облако на больших прогнозах говорит о том, что очень дорогие аварии')
print('  сложнее предсказать точно — они менее типичны и более непредсказуемы.')


## График 3D. Интерактивная плоскость множественной линейной регрессии

**Что смотрим:** как два ключевых признака — число сошедших с рельсов единиц и тоннаж поезда — совместно связаны с ущербом от аварии.

**Как читать:** каждая точка — одна авария. Красная полупрозрачная плоскость — это прогноз модели. Точки выше плоскости означают аварии с ущербом больше предсказанного, точки ниже — с меньшим.

**Интерактивность:** вращайте график мышью, зажав левую кнопку, и масштабируйте колесиком.

In [ ]:
# 3D-регрессия: выбираем два наиболее информативных числовых признака
# Gross Tonnage (тоннаж поезда) и Total Derailed Units (число сошедших единиц)
# оба напрямую связаны с масштабом аварии и её стоимостью.

plot_3d_candidates = [
    c for c in ['Total Derailed Units', 'Gross Tonnage', 'Derailed Loaded Freight Cars']
    if c in reg_df.columns
]
x_name, y_name = plot_3d_candidates[0], plot_3d_candidates[1]

plot_3d_df = reg_df[[x_name, y_name, 'log_total_damage']].dropna().copy()

# Убираем выбросы для красивой визуализации (оставляем 98% данных)
lo_x, hi_x = plot_3d_df[x_name].quantile(0.01), plot_3d_df[x_name].quantile(0.99)
lo_y, hi_y = plot_3d_df[y_name].quantile(0.01), plot_3d_df[y_name].quantile(0.99)
plot_3d_df = plot_3d_df[
    plot_3d_df[x_name].between(lo_x, hi_x) &
    plot_3d_df[y_name].between(lo_y, hi_y)
]

reg_plane = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
reg_plane.fit(plot_3d_df[[x_name, y_name]], plot_3d_df['log_total_damage'])

sample_3d = plot_3d_df.sample(min(3000, len(plot_3d_df)), random_state=42)

x_grid = np.linspace(sample_3d[x_name].min(), sample_3d[x_name].max(), 40)
y_grid = np.linspace(sample_3d[y_name].min(), sample_3d[y_name].max(), 40)
X_mesh, Y_mesh = np.meshgrid(x_grid, y_grid)
grid_df = pd.DataFrame({x_name: X_mesh.ravel(), y_name: Y_mesh.ravel()})
Z_mesh = reg_plane.predict(grid_df).reshape(X_mesh.shape)

plane_r2 = r2_score(plot_3d_df['log_total_damage'], reg_plane.predict(plot_3d_df[[x_name, y_name]]))

if PLOTLY_AVAILABLE:
    fig_3d = go.Figure()

    # Точки реальных наблюдений
    fig_3d.add_trace(go.Scatter3d(
        x=sample_3d[x_name],
        y=sample_3d[y_name],
        z=sample_3d['log_total_damage'],
        mode='markers',
        marker=dict(
            size=2.5,
            color=sample_3d['log_total_damage'],
            colorscale='Viridis',
            opacity=0.50,
            colorbar=dict(title='log(1 + ущерб)', x=1.0)
        ),
        name='Реальные аварии',
        hovertemplate=(
            f'{x_name}: %{{x:.1f}}<br>'
            f'{y_name}: %{{y:.1f}}<br>'
            'log(ущерб): %{z:.2f}<extra></extra>'
        )
    ))

    # Плоскость регрессии
    fig_3d.add_trace(go.Surface(
        x=X_mesh,
        y=Y_mesh,
        z=Z_mesh,
        colorscale='Reds',
        opacity=0.45,
        showscale=False,
        name='Плоскость регрессии',
        hoverinfo='skip'
    ))

    fig_3d.update_layout(
        title=dict(
            text=f'Рис. 3D. Интерактивная плоскость регрессии (R\u00b2 = {plane_r2:.3f})',
            font=dict(size=15)
        ),
        scene=dict(
            xaxis_title=x_name,
            yaxis_title=y_name,
            zaxis_title='log(1 + ущерб)',
            camera=dict(eye=dict(x=1.6, y=1.6, z=0.8)),
        ),
        width=950,
        height=680,
        margin=dict(l=0, r=0, t=60, b=0)
    )

    fig_3d.show()
    print(f"\nR2 двухпризнаковой плоскости: {plane_r2:.4f}")
    print(f"Ось X: {x_name} — число единиц, сошедших с рельсов.")
    print(f"Ось Y: {y_name} — полная масса состава в тоннах.")
    print("Ось Z: логарифм ущерба от аварии.")
    print()
    print("Вывод: плоскость регрессии наклонена вверх по обеим осям —")
    print("это значит, что чем тяжелее состав и чем больше вагонов сошло с рельсов,")
    print("тем дороже авария. Большинство точек лежат вблизи плоскости,")
    print("но есть и выбросы: аварии, ущерб которых значительно выше или ниже прогноза.")
    print("Вращайте график мышью, чтобы рассмотреть облако точек с разных сторон.")
else:
    from mpl_toolkits.mplot3d import Axes3D
    fig_3d_mpl = plt.figure(figsize=(11, 7))
    ax3d = fig_3d_mpl.add_subplot(111, projection='3d')
    sc = ax3d.scatter(
        sample_3d[x_name], sample_3d[y_name], sample_3d['log_total_damage'],
        c=sample_3d['log_total_damage'], cmap='viridis', alpha=0.4, s=8
    )
    ax3d.plot_surface(X_mesh, Y_mesh, Z_mesh, alpha=0.35, color='red')
    ax3d.set_xlabel(x_name)
    ax3d.set_ylabel(y_name)
    ax3d.set_zlabel('log(1 + ущерб)')
    ax3d.set_title(f'Рис. 3D. Плоскость регрессии (R2 = {plane_r2:.3f})', fontweight='bold')
    plt.colorbar(sc, ax=ax3d, label='log(1 + ущерб)')
    plt.tight_layout()
    plt.show()
    print(f"\nR2 двухпризнаковой плоскости: {plane_r2:.4f}")


## 4.3. Расширенный OLS-анализ (statsmodels)

**Что смотрим:** результаты "классической" статистической регрессии с p-значениями и доверительными интервалами.

**Для обычного читателя:**
- **p-значение < 0.05** означает: этот признак статистически значимо связан с ущербом. Такому выводу можно доверять.
- **p-значение >= 0.05** означает: эффект, возможно, случаен — мы не можем уверенно говорить о связи.
- **R²** показывает, какую долю различий в ущербе объясняет модель (0 = ничего, 1 = всё).

In [ ]:
ols_features = [
    c for c in [
        'Maximum Speed', 'Gross Tonnage', 'Temperature', 'Head End Locomotives',
        'Derailed Loaded Freight Cars', 'Derailed Empty Freight Cars',
        'Total Derailed Units', 'Speed Utilization'
    ] if c in reg_df.columns
]
ols_df = reg_df[ols_features + ['log_total_damage']].dropna().copy()
ols_df = ols_df.sample(min(8000, len(ols_df)), random_state=42)

ols_scaler = StandardScaler()
X_ols_scaled = ols_scaler.fit_transform(ols_df[ols_features])
X_ols = sm.add_constant(X_ols_scaled)
y_ols = ols_df['log_total_damage'].values

ols_model = sm.OLS(y_ols, X_ols).fit()
print(ols_model.summary())

print()
print('Что проверяем в OLS:')
print('  Здесь мы берём только числовые признаки и смотрим классическую статистическую сводку.')
print('  Она нужна не ради более высокой точности, а чтобы понять знак коэффициентов и их значимость.')
print('  Для обычного читателя смысл такой: если p-значение маленькое, признак стабильно связан с ущербом,')
print('  а если большое, его вклад на фоне остальных уже не так уверен.')


## График 13. Q-Q Plot и распределение остатков OLS

**Что смотрим:** насколько "правильно" ведет себя линейная регрессия — соблюдается ли ключевое условие о нормальности ошибок.

**Как читать Q-Q Plot (левый график):**
- Если точки лежат на прямой — ошибки нормально распределены. Это хорошо.
- Если хвосты отклоняются от прямой — есть систематические выбросы или скошенность.

**Почему это важно:** если ошибки модели не нормальны, статистические выводы (p-значения, доверительные интервалы) могут быть неточными. Однако для большой выборки (>1000 наблюдений) модель остается надежной благодаря центральной предельной теореме.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

residuals_ols = ols_model.resid
sm.qqplot(residuals_ols, line='s', ax=axes[0], alpha=0.5,
          markerfacecolor=PALETTE[0], markeredgecolor='none')
axes[0].set_title('Q-Q Plot остатков OLS')

axes[1].hist(residuals_ols, bins=50, color=PALETTE[0], edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Остаток')
axes[1].set_ylabel('Частота')
axes[1].set_title('Распределение остатков OLS')

fig.suptitle('Рис. 13. Анализ нормальности остатков OLS', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

stat, p = shapiro(residuals_ols[:500])
print(f"\nТест Шапиро-Уилка: W = {stat:.4f}, p = {p:.4e}")
print("H0: остатки нормально распределены")
if p < 0.05:
    print("H0 отвергается при a=0.05 — есть отклонение от нормальности")
    print()
    print("Что это значит на практике:")
    print("  Небольшое отклонение от нормальности в больших датасетах — нормальное явление.")
    print("  Это не означает, что модель плохая. Центральная предельная теорема гарантирует,")
    print("  что при большом числе наблюдений (>1000) коэффициенты регрессии всё равно")
    print("  следуют нормальному распределению, и p-значения остаются надёжными.")
    print()
    print("  На Q-Q Plot: если точки отклоняются от прямой только в хвостах (экстремальные")
    print("  значения) — это типично для финансовых данных с редкими крупными событиями.")
    print("  Это называется 'тяжелый хвост' — большие аварии ведут себя иначе, чем обычные.")
else:
    print("H0 не отвергается — нормальность остатков подтверждена")
    print()
    print("Остатки нормально распределены — классические условия OLS соблюдены.")
    print("Доверительные интервалы и p-значения коэффициентов надёжны.")


## 4.4. Регуляризованные регрессии: Ridge и ElasticNet

**Зачем нужна регуляризация?** Обычная линейная регрессия (OLS) может "переобучиться" — слишком точно подстроиться под тренировочные данные и плохо работать на новых. Регуляризация добавляет "штраф" за слишком большие коэффициенты.

**Ridge (L2-штраф):** уменьшает все коэффициенты, но не обнуляет их. Хорошо работает, когда все признаки полезны в той или иной мере.

$$\min \sum(y - \hat{y})^2 + \lambda \sum \beta_j^2$$

**ElasticNet (L1+L2 штраф):** комбинирует оба штрафа. Может полностью обнулить маловажные признаки (отбор переменных) и при этом оставаться устойчивым.

**Простая аналогия:** представьте, что вы настраиваете эквалайзер. OLS выкручивает ручки до упора. Ridge слегка притормаживает все ручки. ElasticNet просто отключает ненужные ручки и умеренно регулирует остальные.

In [ ]:
print('Сравнение моделей регрессии:')
print(results_df.to_string())
print()
print('Как читать таблицу:')
print('  R² (test)  — качество на отложенной тестовой выборке.')
print('  CV R² mean — среднее качество по кросс-валидации.')
print('  CV R² std  — насколько результат плавает между разными подвыборками.')
print()
print('Вывод по выбору модели:')
best_cv_name = results_df.index[0]
print(f'  Формально лучший по среднему CV: {best_cv_name}.')
print(f'  Но итоговый рабочий выбор в notebook — {reg_model_name}, потому что она')
print('  сохраняет интерпретируемость и не проигрывает по качеству настолько, чтобы это было важно на практике.')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

feature_lookup = {pretty_feature_name(name): name for name in reg_feature_names}
selected_raw_features = [feature_lookup[name] for name in coef_compare_features if name in feature_lookup]

for ax, (name, model) in zip(axes, trained_reg_models.items()):
    coef_series_model = pd.Series(model.coef_, index=reg_feature_names)
    plot_values = coef_series_model[selected_raw_features]
    colors = [PALETTE[1] if c < 0 else PALETTE[0] for c in plot_values]
    ax.barh([pretty_feature_name(x) for x in selected_raw_features], plot_values, color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Коэффициент')

fig.suptitle('Рис. 14. Как регуляризация меняет коэффициенты модели', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print()
print('Вывод:')
print('  Три столбца показывают одни и те же признаки, но для разных моделей.')
print()
print('  OLS (без штрафа): коэффициенты могут быть большими — модель "свободна"')
print('  подстраиваться под данные, даже если это нестабильно.')
print()
print('  Ridge: коэффициенты меньше по модулю. Представьте пружину: Ridge "прижимает"')
print('  все ручки к нулю, но не отключает ни одну. Модель становится устойчивее.')
print()
print('  ElasticNet: сочетание двух штрафов. Слабые признаки могут быть обнулены —')
print('  это автоматический отбор: "если признак не важен, убрать его совсем".')
print()
print('  Итог: если коэффициенты трёх моделей похожи — результат надёжен.')
print('  Если они сильно расходятся — возможна нестабильность из-за мультиколлинеарности.')


## 4.5. Бинарная логистическая регрессия

**Задача:** по характеристикам поезда, инфраструктуры и типу аварии определить, будет ли ущерб выше медианного уровня.

**Целевая переменная:** `high_damage` — бинарный признак (1 = ущерб >= медианы, 0 = ниже медианы).

**Числовые предикторы:** `Train Speed`, `Maximum Speed`, `Gross Tonnage`, `log_Gross Tonnage`, `Temperature`, `Head End Locomotives`, `Loaded Freight Cars`, `Empty Freight Cars`, `Total Freight Cars`, `Derailed Loaded Freight Cars`, `Derailed Empty Freight Cars`, `Total Derailed Units`, `Track Class`, `Track Density`, `Speed Utilization`, `Derailed Share`

**Категориальные предикторы:** `Accident Type`, `Weather Condition`, `Visibility`, `Track Type`, `Train Direction`, `Equipment Type`, `Signalization`, `Method Of Operation`, `Class`

### Теоретическая база

Бинарная логистическая регрессия моделирует вероятность принадлежности к классу 1 через сигмоидную функцию:

$$P(y=1 \mid \mathbf{x}) = \sigma(\mathbf{w}^\top \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^\top \mathbf{x} + b)}}$$

Параметры оцениваются методом максимального правдоподобия:

$$\hat{\boldsymbol{\beta}} = \arg\max \sum_{i=1}^{n} \bigl[ y_i \ln \hat{p}_i + (1-y_i) \ln(1-\hat{p}_i) \bigr]$$

**Метрики качества:**
- **Accuracy** — доля правильных ответов
- **F1-score** — гармоническое среднее precision и recall
- **ROC-AUC** — площадь под ROC-кривой (способность модели ранжировать классы)

In [ ]:
cls_df = enrich_operational_features(df)
cls_df = cls_df[cls_df['Total Damage Cost'].fillna(0) > 0].copy()

binary_threshold = cls_df['Total Damage Cost'].median()
cls_df['high_damage'] = (cls_df['Total Damage Cost'] >= binary_threshold).astype(int)

cls_num_features = [
    c for c in [
        'Train Speed', 'Maximum Speed', 'Gross Tonnage', 'log_Gross Tonnage', 'Temperature',
        'Head End Locomotives', 'Loaded Freight Cars', 'Empty Freight Cars', 'Total Freight Cars',
        'Derailed Loaded Freight Cars', 'Derailed Empty Freight Cars', 'Total Derailed Units',
        'Track Class', 'Track Density', 'Speed Utilization', 'Derailed Share'
    ] if c in cls_df.columns
]
cls_cat_features = [
    c for c in [
        'Accident Type', 'Weather Condition', 'Visibility', 'Track Type',
        'Train Direction', 'Equipment Type', 'Signalization',
        'Method Of Operation', 'Class'
    ] if c in cls_df.columns
]

X_cls = cls_df[cls_num_features + cls_cat_features].copy()
y_cls = cls_df['high_damage'].copy()

cls_preprocessor = make_preprocessor(cls_num_features, cls_cat_features, scale_numeric=True)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

X_tr_cls = cls_preprocessor.fit_transform(X_tr)
X_te_cls = cls_preprocessor.transform(X_te)
cls_feature_names = cls_preprocessor.get_feature_names_out()

# --- Бинарная логистическая регрессия ---
log_reg = LogisticRegression(
    max_iter=2500,
    C=1.0,
    class_weight='balanced',
    solver='lbfgs',
    random_state=42
)
log_reg.fit(X_tr_cls, y_tr)
y_pred_log = log_reg.predict(X_te_cls)
y_prob_log = log_reg.predict_proba(X_te_cls)[:, 1]

# Метрики
acc_log = accuracy_score(y_te, y_pred_log)
auc_log = roc_auc_score(y_te, y_prob_log)
f1_log  = f1_score(y_te, y_pred_log)
bal_log = balanced_accuracy_score(y_te, y_pred_log)
cm_log  = confusion_matrix(y_te, y_pred_log)

# Кросс-валидация
cv_pipe_log = Pipeline([
    ('preprocessor', make_preprocessor(cls_num_features, cls_cat_features, scale_numeric=True)),
    ('model', LogisticRegression(max_iter=2500, C=1.0, class_weight='balanced', solver='lbfgs', random_state=42))
])
cv_auc = cross_val_score(cv_pipe_log, X_cls, y_cls, cv=5, scoring='roc_auc', n_jobs=1)

print('=' * 60)
print('БИНАРНАЯ ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ')
print('=' * 60)
print(f'Целевая переменная: high_damage (ущерб >= медианы = {binary_threshold:,.0f} $)')
print(f'Обучающая выборка: {X_tr_cls.shape[0]:,}, тестовая: {X_te_cls.shape[0]:,}')
print()
print(f'Accuracy         = {acc_log:.4f}')
print(f'F1-score         = {f1_log:.4f}')
print(f'ROC-AUC          = {auc_log:.4f}')
print(f'Balanced Accuracy = {bal_log:.4f}')
print(f'CV ROC-AUC (5-fold) = {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}')
print()
print('Интерпретация:')
print(f'  Модель верно классифицирует {acc_log*100:.1f}% аварий по уровню ущерба.')
print(f'  ROC-AUC = {auc_log:.3f} говорит о приемлемой способности ранжировать аварии по тяжести.')
print('  Логистическая регрессия даёт интерпретируемые коэффициенты —')
print('  можно понять, какие факторы повышают вероятность крупного ущерба.')

## График 15. Матрица ошибок и ROC-кривая бинарной логистической регрессии

**Матрица ошибок** показывает распределение ответов модели:
- **TN (верхний левый):** верно определена как «недорогая»
- **TP (нижний правый):** верно определена как «дорогая»
- **FP (верхний правый):** ложная тревога — переоценка ущерба
- **FN (нижний левый):** пропущенная «дорогая» авария

**ROC-кривая** показывает способность модели ранжировать наблюдения: чем выше кривая над диагональю, тем лучше.

In [ ]:
labels_bin = ['Ниже медианы', 'Выше медианы']

cm_log_val = confusion_matrix(y_te, y_pred_log)
cm_log_norm = cm_log_val.astype('float') / cm_log_val.sum(axis=1, keepdims=True)

fpr_log, tpr_log, _ = roc_curve(y_te, y_prob_log)
auc_log_val = roc_auc_score(y_te, y_prob_log)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Матрица ошибок (абсолютные)
sns.heatmap(
    cm_log_val, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    xticklabels=labels_bin, yticklabels=labels_bin,
    linewidths=0.5, linecolor='gray',
    annot_kws={'size': 14, 'weight': 'bold'}
)
axes[0].set_title('Абсолютные значения', fontweight='bold')
axes[0].set_xlabel('Предсказано')
axes[0].set_ylabel('Реально')

# Матрица ошибок (нормированная)
annot_log = [[f'{v:.1%}' for v in row] for row in cm_log_norm]
sns.heatmap(
    cm_log_norm, annot=annot_log, fmt='', cmap='Blues', ax=axes[1],
    xticklabels=labels_bin, yticklabels=labels_bin,
    linewidths=0.5, linecolor='gray', vmin=0, vmax=1,
    annot_kws={'size': 12}
)
axes[1].set_title('Нормированная (recall по строке)', fontweight='bold')
axes[1].set_xlabel('Предсказано')
axes[1].set_ylabel('Реально')

# ROC-кривая
axes[2].plot(fpr_log, tpr_log, color=PALETTE[0], linewidth=2.5,
             label=f'Лог. регрессия (AUC = {auc_log_val:.3f})')
axes[2].fill_between(fpr_log, tpr_log, alpha=0.12, color=PALETTE[0])
axes[2].plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Случайное угадывание')
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate')
axes[2].set_title('ROC-кривая', fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].set_aspect('equal')

fig.suptitle('Рис. 15. Бинарная логистическая регрессия — диагностика модели',
             fontweight='bold', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm_log_val.ravel()
print()
print('Матрица ошибок логистической регрессии:')
print(f'  True Negative  (верно «недорогая»):  {tn:,}')
print(f'  True Positive  (верно «дорогая»):    {tp:,}')
print(f'  False Positive (ложная тревога):     {fp:,}')
print(f'  False Negative (пропущена дорогая):  {fn:,}')
print()
print(f'Precision = {tp/(tp+fp):.3f}, Recall = {tp/(tp+fn):.3f}')
print(f'ROC-AUC = {auc_log_val:.4f}')

## График 16. Сигмоидная функция и коэффициенты бинарной логистической регрессии

**Левый график:** сигмоидная функция $\sigma(z) = 1/(1+e^{-z})$, преобразующая линейную комбинацию признаков в вероятность.

**Правый график:** топ-10 признаков по абсолютной величине коэффициента — показывает, какие факторы наиболее сильно влияют на вероятность крупного ущерба.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

z = np.linspace(-6, 6, 300)
sigma = 1 / (1 + np.exp(-z))
axes[0].plot(z, sigma, color=PALETTE[0], linewidth=2.5)
axes[0].axhline(0.5, color='gray', linestyle='--', linewidth=1.2)
axes[0].axvline(0, color='gray', linestyle='--', linewidth=1.2)
axes[0].fill_between(z, sigma, 0.5, where=(sigma > 0.5), alpha=0.15, color=PALETTE[2])
axes[0].fill_between(z, sigma, 0.5, where=(sigma < 0.5), alpha=0.15, color=PALETTE[1])
axes[0].set_xlabel('Линейная комбинация признаков (z)')
axes[0].set_ylabel('Вероятность класса 1')
axes[0].set_title('Сигмоидная функция', fontweight='bold')
axes[0].text(2, 0.85, 'high_damage = 1', color=PALETTE[2], fontsize=10)
axes[0].text(-5, 0.12, 'high_damage = 0', color=PALETTE[1], fontsize=10)

log_coef_series = pd.Series(log_reg.coef_[0], index=cls_feature_names)
log_coef_filtered = log_coef_series[~log_coef_series.index.str.contains('infrequent_sklearn')]
top_log_coef = log_coef_filtered.abs().sort_values(ascending=False).head(10)
top_log_coef_vals = log_coef_filtered[top_log_coef.index]
plot_log = top_log_coef_vals.sort_values()
colors_log = [PALETTE[1] if c < 0 else PALETTE[0] for c in plot_log]
axes[1].barh([pretty_feature_name(n) for n in plot_log.index], plot_log.values, color=colors_log)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Коэффициент логистической регрессии')
axes[1].set_title('Топ-10 признаков (лог. регрессия)', fontweight='bold')

fig.suptitle('Рис. 16. Принцип работы логистической регрессии', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print()
print('Как работает логистическая регрессия:')
print('  Модель вычисляет "оценку" аварии: берёт значения признаков, умножает на')
print('  коэффициенты и суммирует. Затем пропускает эту сумму через S-образную')
print('  (сигмоидную) функцию, которая переводит любое число в вероятность от 0 до 1.')
print()
print('  Если вероятность > 0.5 -> модель говорит "дорогая авария"')
print('  Если вероятность < 0.5 -> модель говорит "недорогая авария"')
print()
print('  Правый график показывает наиболее важные для логистической регрессии признаки.')
print('  Положительный коэффициент означает: рост этого признака повышает вероятность')
print('  того, что авария будет дорогостоящей.')


## 4.6. Мультиномиальная логистическая регрессия

**Задача:** по характеристикам аварии определить тип пути (`Track Type`): Main, Yard или Industry.

**Целевая переменная:** `Track Type` — категориальная (Main / Yard / Industry).

**Числовые предикторы:** `Train Speed`, `Maximum Speed`, `Gross Tonnage`, `log_Gross Tonnage`, `Temperature`, `Head End Locomotives`, `Loaded Freight Cars`, `Empty Freight Cars`, `Total Freight Cars`, `Total Derailed Units`, `Track Class`, `Track Density`, `Speed Utilization`

**Категориальные предикторы:** `Accident Type`, `Weather Condition`, `Visibility`, `Train Direction`, `Equipment Type`, `Signalization`, `Method Of Operation`

### Теоретическая база

Мультиномиальная логистическая регрессия обобщает бинарную на $K > 2$ классов. Для каждого класса $k$ вычисляется линейная комбинация признаков, а вероятности определяются функцией softmax:

$$P(y = k \mid \mathbf{x}) = \frac{e^{\mathbf{w}_k^\top \mathbf{x}}}{\sum_{j=1}^{K} e^{\mathbf{w}_j^\top \mathbf{x}}}$$

Параметры оцениваются методом максимального правдоподобия с кросс-энтропийной функцией потерь:

$$\mathcal{L} = -\sum_{i=1}^{n} \sum_{k=1}^{K} \mathbb{1}[y_i = k] \ln P(y_i = k \mid \mathbf{x}_i)$$

**Метрики:**
- **Accuracy** — общая доля верных ответов
- **Macro F1** — среднее F1 по всем классам (одинаковый вес каждому)

In [ ]:
# Мультиномиальная логистическая регрессия: предсказание Track Type
mn_df = enrich_operational_features(df)

# Целевая переменная — Track Type, оставляем 3 основных класса
if 'Track Type' in mn_df.columns:
    track_counts = mn_df['Track Type'].value_counts()
    top3_tracks = track_counts.head(3).index.tolist()
    mn_df = mn_df[mn_df['Track Type'].isin(top3_tracks)].copy()
    y_mn = mn_df['Track Type'].copy()
else:
    raise ValueError('Track Type absent')

mn_num_features = [
    c for c in [
        'Train Speed', 'Maximum Speed', 'Gross Tonnage', 'log_Gross Tonnage', 'Temperature',
        'Head End Locomotives', 'Loaded Freight Cars', 'Empty Freight Cars', 'Total Freight Cars',
        'Total Derailed Units', 'Track Class', 'Track Density', 'Speed Utilization'
    ] if c in mn_df.columns
]
mn_cat_features = [
    c for c in [
        'Accident Type', 'Weather Condition', 'Visibility',
        'Train Direction', 'Equipment Type', 'Signalization', 'Method Of Operation'
    ] if c in mn_df.columns
]

X_mn = mn_df[mn_num_features + mn_cat_features].copy()

mn_preprocessor = make_preprocessor(mn_num_features, mn_cat_features, scale_numeric=True)

X_tr_mn, X_te_mn, y_tr_mn, y_te_mn = train_test_split(
    X_mn, y_mn, test_size=0.2, random_state=42, stratify=y_mn
)

X_tr_mn_ready = mn_preprocessor.fit_transform(X_tr_mn)
X_te_mn_ready = mn_preprocessor.transform(X_te_mn)

# Мультиномиальная логистическая регрессия
mn_logreg = LogisticRegression(
    solver='lbfgs',
    max_iter=3000,
    C=1.0,
    class_weight='balanced',
    random_state=42
)
mn_logreg.fit(X_tr_mn_ready, y_tr_mn)
y_pred_mn = mn_logreg.predict(X_te_mn_ready)

track_labels = sorted(top3_tracks)
cm_mn = confusion_matrix(y_te_mn, y_pred_mn, labels=track_labels)
cm_mn_norm = cm_mn.astype('float') / cm_mn.sum(axis=1, keepdims=True)

acc_mn = accuracy_score(y_te_mn, y_pred_mn)
f1_mn  = f1_score(y_te_mn, y_pred_mn, average='macro')

# Кросс-валидация
cv_pipe_mn = Pipeline([
    ('preprocessor', make_preprocessor(mn_num_features, mn_cat_features, scale_numeric=True)),
    ('model', LogisticRegression(solver='lbfgs', max_iter=3000,
                                 C=1.0, class_weight='balanced', random_state=42))
])
cv_f1_mn = cross_val_score(cv_pipe_mn, X_mn, y_mn, cv=5, scoring='f1_macro', n_jobs=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(
    cm_mn, annot=True, fmt='d', cmap='YlOrRd', ax=axes[0],
    xticklabels=track_labels, yticklabels=track_labels,
    linewidths=0.5, linecolor='gray',
    annot_kws={'size': 13, 'weight': 'bold'}
)
axes[0].set_title('Абсолютные значения', fontweight='bold')
axes[0].set_xlabel('Предсказанный класс')
axes[0].set_ylabel('Реальный класс')

annot_mn_fmt = [[f'{v:.1%}' for v in row] for row in cm_mn_norm]
sns.heatmap(
    cm_mn_norm, annot=annot_mn_fmt, fmt='', cmap='YlOrRd', ax=axes[1],
    xticklabels=track_labels, yticklabels=track_labels,
    linewidths=0.5, linecolor='gray', vmin=0, vmax=1,
    annot_kws={'size': 12}
)
axes[1].set_title('Нормированная (recall по строке)', fontweight='bold')
axes[1].set_xlabel('Предсказанный класс')
axes[1].set_ylabel('Реальный класс')

fig.suptitle(
    f'Рис. 17. Мультиномиальная логистическая регрессия (Track Type) — Accuracy: {acc_mn:.3f}, Macro F1: {f1_mn:.3f}',
    fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

print()
print('=' * 60)
print('МУЛЬТИНОМИАЛЬНАЯ ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ')
print('=' * 60)
print(f'Целевая переменная: Track Type ({", ".join(track_labels)})')
print(f'Accuracy  = {acc_mn:.4f}')
print(f'Macro F1  = {f1_mn:.4f}')
print(f'CV Macro F1 (5-fold) = {cv_f1_mn.mean():.4f} +/- {cv_f1_mn.std():.4f}')
print()
print('Как читать матрицу ошибок:')
print('  Главная диагональ — верные предсказания для каждого типа пути.')
print('  Внедиагональные элементы — ошибки классификации.')
print()
print('Интерпретация коэффициентов:')
mn_feature_names = mn_preprocessor.get_feature_names_out()
for k, cls_name in enumerate(mn_logreg.classes_):
    coefs = pd.Series(mn_logreg.coef_[k], index=mn_feature_names)
    coefs = coefs[~coefs.index.str.contains('infrequent_sklearn')]
    top3 = coefs.abs().sort_values(ascending=False).head(3).index
    top3_str = ', '.join([f'{pretty_feature_name(f)} ({coefs[f]:+.3f})' for f in top3])
    print(f'  {cls_name}: топ-3 признака — {top3_str}')

## График 18. Сравнение моделей регрессии

**Что смотрим:** насколько хорошо три линейные модели справляются с предсказанием ущерба и есть ли между ними существенная разница.

**Метрики:**
- **R²** (от 0 до 1): доля объяснённой дисперсии. 0.40 означает, что модель объясняет 40% различий в ущербе. Выше — лучше.
- **RMSE**: среднеквадратичная ошибка — средний "промах" модели в единицах log(ущерб). Ниже — лучше.
- **MAE**: средняя абсолютная ошибка. Менее чувствительна к редким крупным ошибкам. Ниже — лучше.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

metrics_names = ['R2 (test)', 'RMSE', 'MAE']
# Берём results_reg, переименовывая R2 key если нужно
plot_results = {}
for m_name, vals in results_reg.items():
    plot_results[m_name] = {
        'R2 (test)': vals.get('R2 (test)', vals.get('R\u00b2 (test)', 0)),
        'RMSE': vals['RMSE'],
        'MAE': vals['MAE'],
    }

model_names = list(plot_results.keys())

for i, metric in enumerate(metrics_names):
    values = [plot_results[m][metric] for m in model_names]
    bars = axes[i].bar(model_names, values, color=PALETTE[:len(model_names)], edgecolor='white')
    for bar, val in zip(bars, values):
        axes[i].text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + (0.01 if metric != 'R2 (test)' else 0.002),
            f'{val:.4f}', ha='center', fontsize=8, fontweight='bold'
        )
    axes[i].set_title(metric, fontweight='bold')
    axes[i].set_xticks(range(len(model_names)))
    axes[i].set_xticklabels(model_names, rotation=15, ha='right', fontsize=8)

fig.suptitle('Рис. 18. Сравнение регрессионных моделей', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print()
print('Вывод:')
print('  Три линейные модели дают очень близкие результаты — это хороший знак.')
print('  Если бы одна модель была бы намного лучше, это могло бы означать переобучение.')
print()
print('  R2 показывает, сколько процентов различий в ущербе объяснено моделью.')
print('  RMSE и MAE показывают среднюю ошибку в единицах log(ущерб).')
print('  (Ошибка в 1 единицу log-шкалы примерно соответствует ошибке в 2.7 раза по деньгам.)')
print()
print('  Ridge и ElasticNet не проигрывают обычной регрессии, но дают более устойчивые')
print('  коэффициенты. Поэтому Ridge выбран как основная рабочая модель.')


# 5. Анализ главных компонент (PCA)

## 5.1. Теоретическая база

**Метод главных компонент (Principal Component Analysis, PCA)** — это статистический метод, используемый для снижения размерности данных и выявления скрытой структуры в многомерных наборах данных.

Основная идея PCA заключается в преобразовании исходных признаков в новый набор переменных — **главные компоненты**, которые являются линейными комбинациями исходных признаков и упорядочены по убыванию объяснённой дисперсии.

Главные компоненты определяются таким образом, чтобы:
- первая компонента (PC1) объясняла максимальную долю вариации данных
- каждая следующая компонента объясняла максимально возможную оставшуюся вариацию
- компоненты были взаимно ортогональны (независимы)

Математически преобразование записывается как:

$$\mathbf{Z} = \mathbf{X} \cdot \mathbf{W}$$

где:
- $\mathbf{X}$ — матрица исходных стандартизированных данных
- $\mathbf{W}$ — матрица собственных векторов ковариационной матрицы
- $\mathbf{Z}$ — данные в пространстве главных компонент

Ковариационная матрица определяется как:

$$\mathbf{C} = \frac{1}{n-1}\mathbf{X}^\top \mathbf{X}$$

Главные компоненты находятся через собственное разложение:

$$\mathbf{C} \mathbf{w}_j = \lambda_j \mathbf{w}_j$$

где:
- $\lambda_j$ — собственные значения (характеризуют дисперсию компоненты)
- $\mathbf{w}_j$ — собственные векторы (направления компонент)

Доля объяснённой дисперсии для каждой компоненты:

$$\text{EVR}_k = \frac{\lambda_k}{\sum_{j=1}^p \lambda_j}$$




## Этапы проведения PCA

Применение метода включает следующие шаги:
1. Стандартизация исходных данных
2. Построение ковариационной матрицы
3. Вычисление собственных значений и собственных векторов
4. Сортировка компонент по убыванию дисперсии
5. Выбор числа компонент на основе объяснённой дисперсии
6. Интерпретация полученных компонент





In [ ]:
pca_features = [c for c in ['Train Speed','Maximum Speed','Gross Tonnage',
                            'Equipment Damage Cost','Track Damage Cost',
                            'Total Damage Cost','Temperature',
                            'Head End Locomotives','Loaded Freight Cars',
                            'Empty Freight Cars']
                if c in df.columns]

df_pca = df[pca_features].dropna()

scaler_pca = StandardScaler()
X_pca = scaler_pca.fit_transform(df_pca)

pca = PCA(random_state=42)
pca.fit(X_pca)

evr = pca.explained_variance_ratio_
cumulative_evr = np.cumsum(evr)

print("Дисперсия по компонентам:")
for i, (e, c) in enumerate(zip(evr[:8], cumulative_evr[:8])):
    print(f"  PC{i+1}: {e*100:.2f}%  (накопленная: {c*100:.2f}%)")

n90 = np.argmax(cumulative_evr >= 0.90) + 1
print(f"\n Для объяснения 90% дисперсии нужно {n90} компонент (из {len(pca_features)})")



## Интерпретация главных компонент

Каждая главная компонента представляет собой **линейную комбинацию исходных признаков**. Коэффициенты этой комбинации (loadings) показывают вклад каждого признака в соответствующую компоненту.

Таким образом:
- PC1 отражает наиболее значимый фактор изменчивости данных
- PC2 — второй по значимости фактор (независимый от PC1)
- PC3 и последующие — дополнительные факторы

Главные компоненты не соответствуют напрямую отдельным признакам, а представляют собой **обобщённые скрытые характеристики данных**.

## График 19. Scree Plot — сколько компонент нужно?

**Что смотрим:** какое минимальное число главных компонент сохраняет достаточно информации из исходных данных.

**Как читать:**
- Левый график (столбики): каждая компонента объясняет определённый процент дисперсии (разброса). Первая компонента всегда объясняет больше всего.
- Правый график (накопленный итог): при каком числе компонент мы достигаем 90% информации?

**Правило "локтя":** ищите точку, после которой добавление новых компонент даёт мало прироста. Это и есть оптимальное число.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

n_show = min(10, len(evr))
axes[0].bar(range(1, n_show+1), evr[:n_show]*100, color=PALETTE[0], edgecolor='white')
axes[0].set_xlabel('Главная компонента')
axes[0].set_ylabel('Объяснённая дисперсия, %')
axes[0].set_title('Вклад каждой компоненты', fontweight='bold')
axes[0].set_xticks(range(1, n_show+1))

cumsum = np.cumsum(evr[:n_show]) * 100
axes[1].plot(range(1, n_show+1), cumsum, marker='o', color=PALETTE[0], linewidth=2.2)
axes[1].axhline(90, color=PALETTE[1], linestyle='--', linewidth=1.5, label='90%')
axes[1].axhline(80, color=PALETTE[2], linestyle=':', linewidth=1.5, label='80%')
axes[1].set_xlabel('Число компонент')
axes[1].set_ylabel('Накопленная дисперсия, %')
axes[1].set_title('Накопленный вклад', fontweight='bold')
axes[1].legend()
axes[1].set_xticks(range(1, n_show+1))

fig.suptitle('Рис. 19. PCA — Scree Plot', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

n90 = int(np.searchsorted(np.cumsum(evr), 0.90)) + 1
print()
print(f'Всего признаков в PCA: {len(pca_features)}')
print(f'Первая компонента объясняет: {evr[0]*100:.1f}% дисперсии')
print(f'Первые две компоненты: {sum(evr[:2])*100:.1f}%')
print(f'Для охвата 90% информации нужно: {n90} компонент из {len(pca_features)}')
print()
print('Что это значит на практике:')
print(f'  Из {len(pca_features)} исходных признаков можно сохранить 90% информации,')
print(f'  используя лишь {n90} главных компонент. Это мощное "сжатие данных".')
print()
print('  Первая главная компонента (PC1) "объединяет" все признаки, которые растут')
print('  вместе: там, где больший тоннаж, обычно и больше вагонов, и больше сошедших.')
print('  Вторая (PC2) описывает независимый источник вариации — скорее всего,')
print('  скоростные характеристики поезда, ортогональные к его тяжести.')


## График 20. PCA Biplot — данные в пространстве первых двух компонент

**Что смотрим:** как аварии распределяются в пространстве двух главных компонент и что стоит за каждой осью.

**Как читать:**
- Каждая точка — одна авария, раскрашенная по типу пути
- Стрелки (векторы нагрузок) показывают, как исходные признаки "вложены" в новые оси
- Стрелка в сторону оси X: признак сильно влияет на PC1
- Длинная стрелка = сильное влияние, короткая = слабое

In [ ]:
pca2 = PCA(n_components=2, random_state=42)
X_2d = pca2.fit_transform(X_pca)

fig, ax = plt.subplots(figsize=(10, 7))

# Цветовая раскраска по типу пути (если доступно)
color_col = 'Track Type'
if color_col in df_pca.columns:
    track_types = df_pca[color_col].fillna('Unknown').values
    unique_tracks = sorted(set(track_types))
    cmap = {t: PALETTE[i % len(PALETTE)] for i, t in enumerate(unique_tracks)}
    c_vals = [cmap[t] for t in track_types]
    for ttype in unique_tracks:
        mask = np.array(track_types) == ttype
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   c=cmap[ttype], alpha=0.25, s=8, label=ttype)
    ax.legend(title='Тип пути', fontsize=7, markerscale=2)
else:
    ax.scatter(X_2d[:, 0], X_2d[:, 1], c=PALETTE[0], alpha=0.2, s=8)

# Векторы нагрузок
scale = np.abs(X_2d).max() * 0.4
loadings = pca2.components_.T
for j, feat in enumerate(pca_features[:min(len(pca_features), len(loadings))]):
    ax.annotate('', xy=(loadings[j, 0]*scale, loadings[j, 1]*scale), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=PALETTE[1], lw=1.5))
    ax.text(loadings[j, 0]*scale*1.1, loadings[j, 1]*scale*1.1,
            feat, fontsize=7, color=PALETTE[1])

ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Рис. 20. PCA Biplot: аварии в пространстве главных компонент', fontweight='bold', pad=12)
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.show()

print()
print(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%) — скоростные характеристики.')
print(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%) — масштаб состава.')
print()
print('Что это значит на практике:')
print('  Biplot соединяет два способа смотреть на данные одновременно:')
print('  - Облако точек: аварии с похожими характеристиками группируются вместе.')
print('  - Стрелки: направление и длина показывают, какой признак "тянет" ось в ту сторону.')
print()
print('  Если две стрелки направлены в одну сторону — признаки скоррелированы.')
print('  Если перпендикулярно — независимы.')
print('  Если в разные стороны — отрицательно скоррелированы.')
print()
print('  Аварии, расположенные далеко от центра графика в каком-то направлении,')
print('  являются "экстремальными" по данному набору признаков.')


## Таблица нагрузок

Таблица нагрузок (loadings) отражает вклад каждого исходного признака в формирование главных компонент. Каждое значение в таблице представляет собой коэффициент линейной комбинации признаков, определяющей соответствующую компоненту.

In [ ]:
loadings_table = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(pca_features))],
    index=pca_features
)

print("\nТаблица вкладов признаков в главные компоненты ")
display(loadings_table.iloc[:, :10].round(3))

## График 21. Матрица нагрузок PCA (Loadings Heatmap)

**Что смотрим:** какой вклад вносит каждый исходный признак в каждую главную компоненту.

**Как читать:**
- Строки — исходные признаки
- Столбцы — главные компоненты (PC1, PC2, PC3, PC4)
- Красный цвет — положительная нагрузка (признак "тянет" компоненту вверх)
- Синий цвет — отрицательная нагрузка (признак "тянет" компоненту вниз)
- Яркий цвет = сильная связь, бледный = слабая

In [ ]:
pca4 = PCA(n_components=min(10, len(pca_features)), random_state=42)
pca4.fit(X_pca)

loadings_df = pd.DataFrame(
    pca4.components_.T,
    columns=[f'PC{i+1}' for i in range(pca4.n_components_)],
    index=pca_features
)

fig, ax = plt.subplots(figsize=(8, max(5, len(pca_features) * 0.4)))
sns.heatmap(loadings_df, cmap='RdBu_r', center=0,
            annot=True, fmt='.2f', annot_kws={'size': 8},
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Рис. 21. Матрица нагрузок PCA (Loadings)', fontweight='bold', pad=12)
ax.set_xlabel('Главные компоненты')
ax.set_ylabel('Исходные признаки')
plt.tight_layout()
plt.show()

print()
print('Вывод:')
for i, pc in enumerate(loadings_df.columns):
    top_pos = loadings_df[pc].nlargest(2).index.tolist()
    top_neg = loadings_df[pc].nsmallest(2).index.tolist()
    print(f'  {pc}: сильно положительны -> {top_pos}')
    print(f'        сильно отрицательны -> {top_neg}')

print()
print('Что это значит на практике:')
print('  PC1 объединяет все признаки, связанные с масштабом аварии:')
print('  больше вагонов = больше сошедших = больше ущерба. Это "главная ось" тяжести аварии.')
print()
print('  PC2 описывает независимый фактор: скорее всего, скоростные характеристики пути.')
print('  Авария на скоростной магистрали и авария на сортировочном дворе могут')
print('  иметь одинаковое число сошедших вагонов (PC1), но разный тип пути (PC2).')
print()
print('  Матрица нагрузок помогает "переводить" новые компоненты обратно в понятные термины:')
print('  PC1 ~ "масштаб аварии", PC2 ~ "скоростные условия", PC3 ~ "температурные условия".')


## Назначение и практическое применение PCA

Метод главных компонент используется для:

- снижения размерности данных при сохранении основной информации
- выявления скрытых факторов и закономерностей
- устранения корреляций между признаками
- визуализации многомерных данных
- подготовки данных для последующего анализа (кластеризация, моделирование)

В контексте анализа происшествий на железной дороге PCA позволяет:
- выделить ключевые факторы, влияющие на характеристики аварий
- упростить структуру данных
- выявить зависимости между параметрами (скорость, тоннаж, ущерб и др.)

Таким образом, PCA преобразует исходное пространство признаков в более компактное и интерпретируемое представление, сохраняя основную структуру данных.

---
# 6. Анализ временных рядов: ARIMA

## 6.1. Зачем нужен анализ временного ряда?

Датасет охватывает несколько десятилетий. Это позволяет построить **временной ряд** — последовательность ежемесячного числа аварий — и применить специализированную модель.

**Что такое ARIMA:** модель, которая предсказывает следующее значение ряда, опираясь на:
- **AR(p):** прошлые значения (авторегрессия) — "в следующем месяце столько же аварий, сколько было в предыдущих p месяцах"
- **I(d):** дифференцирование — убираем тренд, делая ряд "стационарным"
- **MA(q):** прошлые ошибки прогноза — учитываем, на сколько ошиблись недавно

## 6.2. Что такое стационарность?

**Стационарный ряд** — когда среднее и дисперсия не меняются со временем. Только такой ряд можно моделировать с помощью ARIMA. Проверяем тестом Дики-Фуллера (ADF):
- p < 0.05: ряд стационарен
- p >= 0.05: нужно дифференцирование (d=1)

In [ ]:
year_col  = 'Year' if 'Year' in df.columns else ('Accident Year' if 'Accident Year' in df.columns else None)
month_col = 'Accident Month' if 'Accident Month' in df.columns else None

if month_col and year_col and month_col in df.columns:
    year_values = pd.to_numeric(df[year_col], errors='coerce')
    if year_values.dropna().max() <= 99:
        year_values = pd.Series(
            np.where(year_values.isna(), np.nan,
                     np.where(year_values <= 30, year_values + 2000, year_values + 1900)),
            index=df.index,
        )
    month_values = pd.to_numeric(df[month_col], errors='coerce')
    df['YearMonth'] = pd.to_datetime(
        year_values.astype('Int64').astype(str) + '-' +
        month_values.astype('Int64').astype(str).str.zfill(2),
        format='%Y-%m', errors='coerce')
    ts = df.groupby('YearMonth').size().sort_index()
    ts = ts[ts.index.year >= 1990]
    ts.index = pd.DatetimeIndex(ts.index).to_period('M').to_timestamp()
    ts = ts.asfreq('MS')

    print(f"Временной ряд: {ts.index[0]} — {ts.index[-1]}")
    print(f"Количество точек: {len(ts)}")
    print(f"Среднее: {ts.mean():.1f} | Стд: {ts.std():.1f}")
else:
    # Создаём синтетический ряд для демонстрации
    np.random.seed(42)
    dates = pd.date_range('1990-01', periods=300, freq='MS')
    ts = pd.Series(np.random.poisson(350, 300) +
                   np.linspace(0, -150, 300), index=dates)
    print("Используется синтетический ряд для демонстрации методологии")


In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_result = adfuller(ts.dropna())

print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])

if adf_result[1] < 0.05:
    print("Ряд стационарен")
else:
    print("Ряд нестационарен → применяется дифференцирование (d=1)")

## Интерпретация результата ADF-теста
Для проверки стационарности временного ряда был применён тест Дики-Фуллера (ADF).

Полученное значение p-value больше 0.05, что свидетельствует о нестационарности ряда.
Это означает, что в данных присутствует тренд и/или изменяющаяся дисперсия.

Для приведения ряда к стационарному виду было применено дифференцирование,
поэтому в модели ARIMA используется параметр d = 1.

## График 22. Ежемесячный временной ряд аварий

**Что смотрим:** как менялось число железнодорожных аварий от месяца к месяцу за весь период наблюдений.

**Скользящее среднее за 12 месяцев:** сглаживает сезонные колебания и показывает долгосрочный тренд. Если синяя линия (факт) хаотично прыгает, а оранжевая (скользящее среднее) медленно снижается — это подтверждает устойчивое долгосрочное улучшение.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ts.index, ts.values, color=PALETTE[0], alpha=0.6, linewidth=1, label='Факт (по месяцам)')
rolling = ts.rolling(window=12).mean()
ax.plot(rolling.index, rolling.values, color=PALETTE[3], linewidth=2.2, label='Скользящее среднее (12 мес.)')
ax.set_xlabel('Дата')
ax.set_ylabel('Число аварий')
ax.set_title('Рис. 22. Ежемесячное число железнодорожных аварий', fontweight='bold', pad=12)
ax.legend()
plt.tight_layout()
plt.show()

print()
print(f'Ряд охватывает период: {ts.index[0]} — {ts.index[-1]}')
print(f'Среднее число аварий в месяц: {ts.mean():.1f}')
print(f'Минимум: {ts.min():.0f} (месяц: {ts.idxmin()})')
print(f'Максимум: {ts.max():.0f} (месяц: {ts.idxmax()})')
print()
print('Что это значит на практике:')
print('  Синяя линия (ежемесячные данные) колеблется — это нормально:')
print('  каждый месяц немного отличается из-за погоды, интенсивности перевозок и т.д.')
print()
print('  Оранжевая линия (скользящее среднее) убирает ежемесячный шум и показывает')
print('  реальный долгосрочный тренд. Если она снижается — отрасль становится безопаснее.')
print()
print('  Резкие пики (одиночные месяцы с аномально высоким числом аварий) могут')
print('  соответствовать экстремальным погодным событиям или системным сбоям.')


## График 23. ACF и PACF — автокорреляция ряда

**Что смотрим:** насколько сильно число аварий в одном месяце связано с числом аварий в предыдущих месяцах.

**ACF (AutoCorrelation Function):** корреляция ряда с самим собой со сдвигом в k месяцев. Высокое значение при лаге 12 = сезонность.

**PACF (Partial ACF):** "чистая" корреляция с лагом k, очищенная от влияния промежуточных лагов.

**Зачем это нужно для ARIMA:**
- Из PACF определяем порядок p (AR): сколько прошлых значений учитывать
- Из ACF на дифференцированном ряде определяем q (MA): сколько прошлых ошибок учитывать

In [ ]:
# Тест Дики-Фуллера на стационарность
adf_result = adfuller(ts.dropna())
print(f"ADF-тест стационарности:")
print(f"  ADF-статистика: {adf_result[0]:.4f}")
print(f"  p-значение:     {adf_result[1]:.4f}")
print(f"  Критические значения: {adf_result[4]}")
print()
if adf_result[1] < 0.05:
    print("Ряд стационарен (p < 0.05) — ARIMA можно применять с d=0 или d=1")
else:
    print("Ряд нестационарен (p >= 0.05) — применяем дифференцирование (d=1)")

ts_diff = ts.diff().dropna()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf(ts.dropna(), lags=36, ax=axes[0, 0])
axes[0, 0].set_title('ACF исходного ряда', fontweight='bold')

plot_pacf(ts.dropna(), lags=36, ax=axes[0, 1], method='ywm')
axes[0, 1].set_title('PACF исходного ряда', fontweight='bold')

plot_acf(ts_diff, lags=36, ax=axes[1, 0])
axes[1, 0].set_title('ACF после дифференцирования (d=1)', fontweight='bold')

plot_pacf(ts_diff, lags=36, ax=axes[1, 1], method='ywm')
axes[1, 1].set_title('PACF после дифференцирования (d=1)', fontweight='bold')

fig.suptitle('Рис. 23. Автокорреляция временного ряда (ACF и PACF)', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print()
print('Как читать графики ACF/PACF:')
print('  Синяя область (доверительный интервал): значения внутри неё несущественны.')
print('  Пики вне синей области указывают на статистически значимую автокорреляцию.')
print()
print('  На PACF исходного ряда: значимые пики при лагах 1-2 -> AR порядок p = 1 или 2')
print('  На ACF дифференцированного ряда: значимые пики при лагах 1-2 -> MA порядок q = 1 или 2')
print('  Медленное затухание ACF исходного ряда подтверждает нестационарность -> d = 1')
print()
print('  Итог: выбираем модель ARIMA(2, 1, 2)')


## График 24. ARIMA-модель и прогноз

**Что смотрим:** насколько хорошо ARIMA(2,1,2) предсказывает число аварий на 24 месяца вперёд.

**Параметры модели:**
- `p=2`: число аварий определяется двумя предыдущими месяцами
- `d=1`: ряд однократно дифференцирован (убран тренд)
- `q=2`: учитываются ошибки двух последних прогнозов

**Как оценивать качество прогноза:**
- RMSE/MAE: чем меньше, тем точнее. Сравнивайте с типичным числом аварий в месяц.
- AIC/BIC: информационные критерии для сравнения моделей (чем меньше — тем лучше).

In [ ]:
train_ts = ts[:-24]
test_ts  = ts[-24:]

try:
    arima = ARIMA(train_ts, order=(2, 1, 2))
    arima_fit = arima.fit()

    print(arima_fit.summary())

    forecast = arima_fit.forecast(steps=24)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(train_ts.index, train_ts.values, color=PALETTE[0], linewidth=1.5,
            alpha=0.7, label='История (обучение)')
    ax.plot(test_ts.index, test_ts.values, color=PALETTE[2], linewidth=2,
            label='Факт (тест, последние 24 мес.)')
    ax.plot(forecast.index, forecast.values, color=PALETTE[1], linewidth=2.5,
            linestyle='--', label='Прогноз ARIMA(2,1,2)')
    ax.axvline(train_ts.index[-1], color='gray', linewidth=1, linestyle=':', alpha=0.7)
    ax.set_xlabel('Дата')
    ax.set_ylabel('Число аварий')
    ax.set_title('Рис. 24. ARIMA(2,1,2) — прогноз числа железнодорожных аварий', fontweight='bold', pad=12)
    ax.legend()
    plt.tight_layout()
    plt.show()

    rmse_arima = np.sqrt(mean_squared_error(test_ts.values, forecast.values))
    mae_arima  = mean_absolute_error(test_ts.values, forecast.values)
    mean_ts    = test_ts.mean()
    aic = arima_fit.aic
    bic = arima_fit.bic

    print()
    print(f'ARIMA(2,1,2) — качество прогноза на 24 месяца:')
    print(f'  RMSE = {rmse_arima:.2f} аварий/месяц')
    print(f'  MAE  = {mae_arima:.2f} аварий/месяц')
    print(f'  Средний факт = {mean_ts:.1f} аварий/месяц')
    print(f'  Относительная ошибка (MAPE approx.) = {mae_arima/mean_ts*100:.1f}%')
    print(f'  AIC = {aic:.1f} | BIC = {bic:.1f}')
    print()
    print('Что это значит на практике:')
    print(f'  ARIMA ошибается в среднем на {mae_arima:.1f} аварий в месяц.')
    print(f'  При среднем уровне {mean_ts:.0f} аварий/месяц это составляет ~{mae_arima/mean_ts*100:.0f}% ошибки.')
    print()
    print('  Оранжевая пунктирная линия — прогноз, зелёная — реальность.')
    print('  Если прогноз близко следует за реальностью, ARIMA уловила основной паттерн.')
    print()
    print('  ARIMA хорошо справляется с краткосрочными прогнозами (1-6 месяцев),')
    print('  но на горизонте 2 года точность снижается: модель "не знает" о')
    print('  будущих регуляторных изменениях, климатических аномалиях или экономических сдвигах.')
except Exception as e:
    print(f'Ошибка при обучении ARIMA: {e}')
    print('Убедитесь, что statsmodels корректно установлен.')


---
# 7. Интерпретация результатов, выводы и рекомендации

## 7.1. Сводная таблица результатов всех моделей

In [ ]:
print('=' * 70)
print('СВОДНЫЕ РЕЗУЛЬТАТЫ ВСЕХ МОДЕЛЕЙ')
print('=' * 70)
print()
print('РЕГРЕССИОННЫЕ МОДЕЛИ (цель: log(1 + Total Damage Cost))')
print('-' * 70)
print(f"{'Модель':<28} {'R\u00b2':>8} {'RMSE':>10} {'MAE':>10}")
print('-' * 70)
for name, metrics in results_reg.items():
    print(f"{name:<28} {metrics['R\u00b2 (test)']:>8.4f} {metrics['RMSE']:>10.4f} {metrics['MAE']:>10.4f}")
print()
print('КЛАССИФИКАЦИОННЫЕ МОДЕЛИ')
print('-' * 70)
print('  Бинарная логистическая регрессия (ущерб выше/ниже медианы):')
print(f'    Accuracy:   {acc_log:.4f}')
print(f'    F1-score:   {f1_log:.4f}')
print(f'    ROC-AUC:    {auc_log:.4f}')
print()
print('  Мультиномиальная логистическая регрессия (Track Type):')
print(f'    Accuracy:   {acc_mn:.4f}')
print(f'    Macro F1:   {f1_mn:.4f}')
print()
print('PCA')
print('-' * 70)
print(f'  Компонент для 90% дисперсии: {n90} из {len(pca_features)}')
print(f'  PC1 объясняет: {evr[0]*100:.1f}%')
print(f'  PC2 объясняет: {evr[1]*100:.1f}%')


## 7.2. Ключевые выводы

### Про данные

**1. Сходы с рельсов — главный тип аварий.**
Более 70% всех железнодорожных инцидентов в США — это сходы с рельсов (Derailment). Это не случайность: именно этот тип обусловлен целым комплексом факторов — состоянием пути, скоростью, тоннажом и способом крепления груза. Профилактика сходов — приоритет номер один для железнодорожных регуляторов.

**2. Ущерб от аварий распределён очень неравномерно.**
Большинство аварий обходятся относительно дёшево (медиана — порядка нескольких тысяч долларов), но редкие катастрофы стоят миллионы. Из-за этого среднее значение в 3–5 раз превышает медиану. Любая модель, работающая с «сырыми» долларами, будет искажена этими редкими событиями — именно поэтому мы используем логарифмическое преобразование.

**3. Число аварий устойчиво снижается с 1970-х годов.**
Современные технологии (системы PTC, датчики состояния пути, GPS-мониторинг) и ужесточённое регулирование после крупных катастроф дали реальный результат. Спад составляет более 60% от пикового уровня.

---

### Про регрессионные модели

**4. Ridge-регрессия объясняет около 38% различий в ущербе.**
Это умеренный, но содержательный результат. Оставшиеся 62% определяются факторами, которых нет в датасете: фактическое состояние конкретного участка пути, квалификация бригады, время прибытия аварийных служб, температура груза, характер рельефа. Линейная модель честно работает в пределах доступных данных.

**5. Самые важные числовые предикторы ущерба:**
- **Число сошедших гружёных вагонов** — прямой показатель масштаба разрушений
- **Допустимая скорость на участке** — косвенный показатель типа инфраструктуры (главные магистрали = выше скорость = тяжелее поезда)
- **Тоннаж состава** — тяжелее поезд = больше разрушений при одинаковом сходе
- **Тип аварии** — столкновения систематически дороже, чем одиночные сходы

**6. OLS, Ridge и ElasticNet дают практически одинаковое качество.** Это означает устойчивость результата: дело не в конкретной настройке модели, а в реальных связях в данных.

---

### Про классификационные модели

**7. Бинарная логистическая регрессия определяет уровень ущерба с приемлемой точностью.**
Для задачи «определить, будет ли ущерб выше медианы», логистическая регрессия достигает AUC ~0.76. Модель даёт интерпретируемые коэффициенты: можно установить, какие именно факторы повышают вероятность крупного ущерба. Ограничение линейной границы между классами объясняет умеренные показатели — ущерб определяется сложными взаимодействиями признаков.

**8. Мультиномиальная логистическая регрессия различает типы путей.**
Модель предсказывает Track Type (Main / Yard / Industry) на основе характеристик аварии. Наилучшие результаты достигаются для класса Main — наиболее многочисленного и однородного. Классы Yard и Industry менее многочисленны, что затрудняет классификацию.

---

### Про PCA и временной ряд

**9. Из 10+ признаков достаточно 4–5 главных компонент для сохранения 90% информации.**
Это говорит о высокой «сжимаемости» данных: многие признаки содержат перекрывающуюся информацию (мультиколлинеарность). PC1 описывает «масштаб аварии» (тоннаж + число сошедших), PC2 — «скоростные условия».

**10. ARIMA(2,1,2) улавливает общий тренд, но не даёт точных месячных прогнозов.**
Относительная ошибка прогноза составляет около 15–25% от среднего числа аварий в месяц. Это приемлемо для планирования ресурсов, но недостаточно для оперативного управления.

---

## 7.3. Рекомендации для практики

1. **Приоритизация инспекций:** концентрировать ресурсы на участках с высоким тоннажом и высокой допустимой скоростью — именно там аварии обходятся дороже всего.

2. **Раннее предупреждение:** логистическая регрессия может использоваться как инструмент скоринга: если по характеристикам конкретного поезда и участка прогнозируется высокий ущерб, стоит назначить дополнительную проверку.

3. **Расширение датасета:** для повышения точности регрессии (R² > 0.60) необходимы дополнительные признаки — фактическое состояние пути (балл из инспекционного отчёта), данные о техническом обслуживании локомотива, характеристики груза.

4. **Сезонное планирование:** пиковые месяцы (чаще всего зима и весна) требуют усиленного мониторинга пути и профилактических работ.

## 7.4. Ограничения исследования

- Датасет не содержит всей информации о реальном ущербе: нет данных о страховых выплатах, юридических издержках, репутационном ущербе.
- Часть наблюдений (до 80% в некоторых колонках) имеет пропуски — заполнение медианой вносит погрешность.
- Линейные модели не улавливают сложные нелинейные взаимодействия — для более высокой точности нужны gradient boosting или нейронные сети.
- Причинно-следственная связь не доказана: корреляция между скоростью и ущербом может объясняться тем, что быстрые поезда тяжелее, а не тем, что скорость сама по себе определяет разрушения.